# 11 — Auditoria: memória de cálculo das decisões metodológicas
**TCC — Preços e concentração de fornecedores em compras públicas de insumos hospitalares, 2009–2023**  
Mônica Anatalia Bezerra de Araujo — MBA em Data Science e Analytics para Operações, POLI USP PRO

**Objetivo.** Reunir, num único lugar, os testes que fundamentam cada decisão de tratamento dos dados
tomada em out/2026. Cada seção traz: a decisão, o teste, o número que a sustenta e um arquivo CSV com a
tabela correspondente (pasta `Auditoria`). Nenhuma base é alterada por esta rotina.

| Seção | Decisão |
|---|---|
| 0 | Correções na leitura da fonte (rotina 03) e conferência independente (rotina 10) |
| 1 | Comprador público definido pela natureza jurídica do CNPJ (Receita Federal, dez/2023) |
| 2 | Exclusão dos registros sem CNPJ do comprador |
| 3 | Recuperação de UF e município pelo CNPJ do comprador |
| 4 | Valores implausíveis (erros de digitação da fonte) |
| 5 | Concentração medida de duas formas: por número de compras e por valor |
| 6 | Mínimo de 7 compras por item e ano para classificar a concentração |
| 7 | Índice de preço real: sensibilidade e fragilidade do início da série |
| 8 | Escala: item definido como CATMAT + unidade e mínimo de 3 compras no item-ano (D17, D53) |
| 9 | Índice territorial: estabilidade e a ressalva de Roraima |
| 10 | Representatividade da base ano a ano |
| 11 | Capacidade dos dados: fabricante, classe do material e medicamentos x materiais |
| 12 | Retirada das duplicatas exatas da fonte (concentradas no arquivo de 2023) |
| 13 | Indicadores do painel (rotina 12): testes que os sustentam, e a comparação com a CMED descartada |
| 14 | Quem domina o item no mapa de oportunidades: valor e número de compras, por item-ano (D28 e D29) |
| 15 | Classe CATMAT e exclusão dos códigos fora da saúde (D40, D41) |
| 16 | Curvas de robustez da fase 2: permanência dos itens (D30), mínimo de anos (D34), ano de início (D47) |
| 17 | Escolha da fonte de dados: Fonte A (bases compiladas) x Fonte B (dados.gov.br) e a ressalva de 2023 (D21) |

**Fase 2 (out/2026).** Os testes passam a usar a **base analítica** (rotina 04B, D14), a mesma das rotinas de análise:
item = CATMAT + unidade (D17), régua sobre o preço real (D16), só as classes de saúde (D41). `pub` é a base inteira
(contagens); `pub_val` são as compras válidas para preço (D16). Seções novas: 6c (faixas de fornecedores, D33),
6d (concentração por CATMAT como teste de robustez, D17), 15 e 16.

**Entradas:** `Base Analitica`, `Base Conversao CSV`, `Base Harmonizacao Campos`, `Base Harmonizacao Publica`,
`Base Classificacao Receita`, `Comparacao Fontes` (seção 17), `IPCA Tratado` e as saídas das rotinas 05 e 09 (seção 16). **Saída:** `Auditoria`.

In [1]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass   # fora do Colab: use a variável de ambiente BPS_DADOS
PASTA_DADOS = os.environ.get('BPS_DADOS', '/content/drive/MyDrive/TCC/Registro de Compras v2/Base v2')
# ─────────────────────────────────────────────────────────────────────────

Mounted at /content/drive


In [2]:
!pip install polars==1.35.2 scipy xlrd openpyxl python-calamine==0.8.3 --quiet   # calamine: leitor de Excel só da seção 17 (D21)

In [3]:
from pathlib import Path
import polars as pl, numpy as np
from scipy.stats import spearmanr
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_cols(-1); pl.Config.set_fmt_str_lengths(60); pl.Config.set_tbl_width_chars(220)

BASE        = Path(PASTA_DADOS)
P_CSV       = BASE / 'Base Conversao CSV'
P_HARM      = BASE / 'Base Harmonizacao Campos'
P_PUB       = BASE / 'Base Harmonizacao Publica'
P_REC       = BASE / 'Base Classificacao Receita'
ARQ_IPCA    = BASE / 'IPCA Tratado' / 'ipca_deflator.parquet'
ARQ_BASE    = BASE / 'Base Analitica' / 'base_analitica.parquet'   # rotina 04B (D14)
SAIDA       = BASE / 'Auditoria'
ANOS        = range(2009, 2024)
SAIDA.mkdir(parents=True, exist_ok=True)
for p in [P_CSV, P_HARM, P_PUB, P_REC, ARQ_IPCA, ARQ_BASE]:
    print(f'{"OK " if p.exists() else "FALTANDO"}  {p}')
assert all(p.exists() for p in [P_CSV, P_HARM, P_PUB, P_REC, ARQ_IPCA, ARQ_BASE]), 'Confira PASTA_DADOS (e rode a rotina 04B)'

def salvar(df, nome):
    df.write_csv(SAIDA / f'{nome}.csv'); print(f'  -> Auditoria/{nome}.csv')

def ler(pasta, colunas):
    fr = []
    for a in ANOS:
        arq = pasta / f'BPS_{a}.parquet'
        disp = pl.read_parquet(arq, n_rows=0).columns
        d = pl.read_parquet(arq, columns=[c for c in colunas if c in disp])
        for c in colunas:
            if c not in d.columns:
                d = d.with_columns(pl.lit(None, pl.Utf8).alias(c))
        fr.append(d.select(colunas).with_columns(pl.lit(a).alias('arquivo')))
    return pl.concat(fr, how='diagonal')

ipca = pl.read_parquet(ARQ_IPCA).select(['ano', 'mes', 'fator_deflacao'])
# Fase 2 (D14): a auditoria lê a MESMA base das rotinas de análise. pub = base inteira (contagens, D16);
# pub_val = compras válidas para preço (sem valor implausível, D16).
REGUA_PRECO, REGUA_QTD, MIN_REG_REGUA = 20, 200, 3   # D16/D7: a régua adotada (as seções 4 e 5 testam outras)
MIN_COMPRAS = 7                                       # D7
LIM_NAO, LIM_ALTO = 1500, 2500                        # D7: faixas do CADE (2016)
MIN_MEDIANA = 3                                       # D45/D53: mediana do item no ano com 3 preços ou mais
CORTE_ITEM_GRANDE = 0.8                               # D29: item grande = compras iguais ou acima do percentil 80 do ano
COLS_BASE = ['data_compra', 'ano', 'mes', 'item', 'cod_catmat', 'unidade_chave', 'desc_item', 'classe_catmat', 'nome_classe',
             'origem_classe', 'preco_unitario', 'quantidade', 'valor_total', 'fator_deflacao', 'preco_real', 'valor_real',
             'valor_implausivel', 'cnpj_fornecedor', 'cnpj_instituicao', 'instituicao', 'uf', 'municipio', 'natureza_desc',
             'unidade_marca_generico', 'modalidade_grupo', 'compras_item_ano', 'compras_validas_item_ano',
             'preco_rel_mediana', 'lote_rel', 'faixa_lote']   # só as colunas usadas (memória do Colab)
pub = pl.read_parquet(ARQ_BASE, columns=COLS_BASE)
pub_val = pub.filter(~pl.col('valor_implausivel').fill_null(False))   # nulo = não avaliável pela régua: fica
print(f'\nBase analítica para auditoria: {pub.height:,} compras | válidas para preço (D16): {pub_val.height:,}')

OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Conversao CSV
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Harmonizacao Campos
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Harmonizacao Publica
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Classificacao Receita
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/IPCA Tratado/ipca_deflator.parquet
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Analitica/base_analitica.parquet

Base analítica para auditoria: 833,928 compras | válidas para preço (D16): 802,660


## 0. Correções na leitura da fonte (rotina 03)
**Decisão.** A rotina 03 corrige problemas de leitura dos arquivos do BPS antes de qualquer análise. Esta seção mostra, ano a ano, o que cada correção encontrou na fonte. As contagens são feitas pela própria rotina 03 sobre o texto original, antes do tratamento, e gravadas na pasta Auditoria (arquivos 00a a 00g).

| Arquivo | Correção |
|---|---|
| 00a | Data da compra: em 2009–2012 vem com espaço invisível; sem limpá-lo, nenhuma data é lida |
| 00b | Separador decimal: ponto só é milhar quando há vírgula (a regra anterior multiplicava os preços) |
| 00c | Valor total: quando o arquivo não traz a coluna, é calculado como preço × quantidade |
| 00d | Colunas de CNPJ: em 2018–2020 os rótulos da fonte vêm trocados ("Fornecedor" traz o CNPJ) |
| 00e | Código CATMAT: padronizado como "BR" + 7 dígitos; código vazio ou de outro tamanho vira vazio |
| 00f | Marca "GENÉRICO" escrita dentro da unidade: sai da unidade e fica em campo próprio (D10) |
| 00g | Linhas sem preço e sem descrição: retiradas |
| 00h | Códigos de 6 dígitos completados com "BR0": quantos reconectam com códigos de outros anos (calculado aqui) |

A conferência independente dessas leituras, feita a partir dos arquivos Excel originais, está na rotina 10 (arquivos 10a a 10d).

In [4]:
for nome, titulo in [('00a_datas_da_compra', 'Datas da compra'),
                     ('00b_separador_decimal', 'Separador decimal (medianas pela regra anterior e pela atual)'),
                     ('00c_valor_total', 'Valor total: da fonte ou calculado'),
                     ('00d_colunas_cnpj', 'Colunas de CNPJ (% de valores numéricos em cada coluna)'),
                     ('00e_codigo_catmat', 'Código CATMAT'),
                     ('00f_marca_generico_na_unidade', 'Marca "GENÉRICO" na unidade'),
                     ('00g_linhas_sem_preco_e_descricao', 'Linhas sem preço e sem descrição')]:
    arq = SAIDA / f'{nome}.csv'
    assert arq.exists(), f'{arq.name} não encontrado: rode a rotina 03 antes'
    print(f'\n{nome} — {titulo} (gravado pela rotina 03):')
    print(pl.read_csv(arq))

# 00h — códigos CATMAT de 6 dígitos completados com "BR0": o código completado existe em outros anos?
t00e = pl.read_csv(SAIDA / '00e_codigo_catmat.csv')
cod = ler(P_HARM, ['cod_catmat']).drop_nulls('cod_catmat')
linhas00h = []
for a in t00e.filter(pl.col('catmat_6_digitos_vira_BR0') > 0)['ano'].to_list():
    do_ano = cod.filter(pl.col('arquivo') == a)
    outros = set(cod.filter(pl.col('arquivo') != a)['cod_catmat'].unique().to_list())
    cods = do_ano['cod_catmat'].unique()
    linhas00h.append({'ano': a, 'codigos_no_ano': cods.len(), 'codigos_vistos_em_outros_anos': int(cods.is_in(list(outros)).sum()),
                      'linhas_no_ano': do_ano.height, 'linhas_com_codigo_visto_em_outros_anos': int(do_ano['cod_catmat'].is_in(list(outros)).sum())})
t00h = pl.DataFrame(linhas00h).with_columns(
    (pl.col('codigos_vistos_em_outros_anos') / pl.col('codigos_no_ano') * 100).round(1).alias('pct_codigos'),
    (pl.col('linhas_com_codigo_visto_em_outros_anos') / pl.col('linhas_no_ano') * 100).round(1).alias('pct_linhas'))
print('\n00h — códigos de 6 dígitos completados com "BR0": reconexão com os demais anos')
print(t00h); salvar(t00h, '00h_catmat_6_digitos_reconexao')

t00a = pl.read_csv(SAIDA / '00a_datas_da_compra.csv')
print(f"\nDatas: {t00a['data_preenchida_fonte'].sum():,} preenchidas na fonte | lidas pela regra atual: {t00a['data_lida_regra_atual'].sum():,} "
      f"| lidas sem limpar o espaço invisível: {t00a['data_lida_sem_limpar_espaco'].sum():,}")
t00f = pl.read_csv(SAIDA / '00f_marca_generico_na_unidade.csv')
print(f'Marca "GENÉRICO" na unidade: {t00f["unidade_com_marca_generico"].sum():,} linhas (base harmonizada, antes do filtro de comprador público)')
for nome in ['10a_conservacao_por_ano', '10b_coerencia_valor_total', '10c_duplicatas_independente', '10d_replicacao_indicadores']:
    arq = SAIDA / f'{nome}.csv'
    print(f'\n{nome} (gravado pela rotina 10):')
    print(pl.read_csv(arq) if arq.exists() else '  ainda não gerado: rode a rotina 10')



00a_datas_da_compra — Datas da compra (gravado pela rotina 03):
shape: (15, 5)
┌──────┬────────────────────────┬───────────────────────┬─────────────────────────────┬───────────────────────┐
│ ano  ┆ linhas_apos_duplicatas ┆ data_preenchida_fonte ┆ data_lida_sem_limpar_espaco ┆ data_lida_regra_atual │
│ ---  ┆ ---                    ┆ ---                   ┆ ---                         ┆ ---                   │
│ i64  ┆ i64                    ┆ i64                   ┆ i64                         ┆ i64                   │
╞══════╪════════════════════════╪═══════════════════════╪═════════════════════════════╪═══════════════════════╡
│ 2009 ┆ 13463                  ┆ 13463                 ┆ 0                           ┆ 13463                 │
│ 2010 ┆ 19066                  ┆ 19066                 ┆ 0                           ┆ 19066                 │
│ 2011 ┆ 13611                  ┆ 13611                 ┆ 0                           ┆ 13611                 │
│ 2012 ┆ 4875           

## 1. Comprador público pela natureza jurídica do CNPJ
**Decisão.** Entram apenas compradores da administração pública direta e indireta (Decreto-Lei nº 200/1967,
art. 4º), identificados pela natureza jurídica do CNPJ no cadastro da Receita Federal de dez/2023 (rotinas 03C e 03B).
**Teste.** Cobertura do cruzamento e conferência da esfera informada pelo próprio BPS (2009–2016).

In [5]:
rec = pl.read_parquet(P_REC / 'compradores_receita.parquet')
print(f'CNPJs compradores: {rec.height:,} | encontrados na Receita: {rec["encontrado_receita"].sum():,} '
      f'| digito verificador invalido: {(~rec["cnpj_valido"]).sum():,}')
conf = (rec.filter(pl.col('esfera_bps').is_not_null())
           .with_columns(pl.col('grupo_natureza').fill_null('NAO ENCONTRADO'))
           .group_by('esfera_bps', 'grupo_natureza').agg(pl.len().alias('cnpjs'), pl.col('registros').sum().alias('registros'))
           .sort(['esfera_bps', 'cnpjs'], descending=[False, True]))
print('\nEsfera informada pelo BPS x grupo de natureza juridica da Receita:')
print(conf)
salvar(conf, '01_esfera_bps_x_natureza')
for arq in ['fluxo_classificacao_comprador.csv', 'transicao_regra_receita.csv', 'excluidos_por_natureza.csv']:
    if (P_PUB / arq).exists():
        print(f'\n{arq} (gerado pela 03B):'); print(pl.read_csv(P_PUB / arq))

CNPJs compradores: 2,081 | encontrados na Receita: 2,081 | digito verificador invalido: 0

Esfera informada pelo BPS x grupo de natureza juridica da Receita:
shape: (8, 4)
┌──────────────────┬───────────────────────────────┬───────┬───────────┐
│ esfera_bps       ┆ grupo_natureza                ┆ cnpjs ┆ registros │
│ ---              ┆ ---                           ┆ ---   ┆ ---       │
│ str              ┆ str                           ┆ u32   ┆ u32       │
╞══════════════════╪═══════════════════════════════╪═══════╪═══════════╡
│ ESTADUAL         ┆ Administracao Publica         ┆ 64    ┆ 43504     │
│ ESTADUAL         ┆ Entidades Empresariais        ┆ 1     ┆ 1         │
│ ESTADUAL/PRIVADA ┆ Entidades sem Fins Lucrativos ┆ 1     ┆ 1651      │
│ FEDERAL          ┆ Administracao Publica         ┆ 19    ┆ 6137      │
│ MUNICIPAL        ┆ Administracao Publica         ┆ 883   ┆ 417829    │
│ PRIVADA          ┆ Entidades sem Fins Lucrativos ┆ 45    ┆ 32433     │
│ PRIVADA          ┆ Admi

## 2. Registros sem CNPJ do comprador
**Decisão.** Excluídos: sem CNPJ não é possível identificar a natureza jurídica.
**Teste.** O vazio vem do arquivo de origem (não do ETL) e o CNPJ não está deslocado para outra coluna.

In [6]:
linhas = []
for a in ANOS:
    df = pl.read_csv(P_CSV / f'BPS_{a}.csv', infer_schema_length=0, encoding='utf8-lossy')
    df = df.unique(maintain_order=True)   # mesma retirada de duplicatas exatas da rotina 03 (secao 12)
    df = df.rename({c: c.replace('\xa0', '').replace('\n', ' ').strip() for c in df.columns})
    col = next(c for c in ['CNPJ Instituição', 'CNPJ Comprador', 'Instituição'] if c in df.columns)
    s = df[col].str.replace_all(r'\D', '')
    vazio = s.is_null() | (s.str.len_chars() == 0) | (s.str.replace_all('0', '') == '')
    nome_col = next((c for c in ['Nome Instituição', 'Instituição', 'Comprador'] if c in df.columns and c != col), None)
    sub = df.filter(vazio)
    sem_nome = (sub[nome_col].str.strip_chars().is_null() | sub[nome_col].str.contains('(?i)n.o informado').fill_null(False)
                | (sub[nome_col].str.strip_chars() == '')).sum() if nome_col and sub.height else sub.height
    linhas.append({'ano': a, 'linhas_arquivo': df.height, 'sem_cnpj_comprador_no_bruto': int(vazio.sum()),
                   'dos_quais_sem_nome': int(sem_nome)})
t2 = pl.DataFrame(linhas)
h = ler(P_HARM, ['cnpj_instituicao'])
sem_h = (h.with_columns(pl.col('cnpj_instituicao').cast(pl.Utf8).str.replace_all(r'\D', '').alias('d'))
          .filter(pl.col('d').is_null() | (pl.col('d').str.len_chars() == 0))
          .group_by('arquivo').len().rename({'arquivo': 'ano', 'len': 'sem_cnpj_na_harmonizada'}))
t2 = t2.join(sem_h, on='ano', how='left').fill_null(0)
print(t2)
print(f'\nTotal sem CNPJ: bruto {t2["sem_cnpj_comprador_no_bruto"].sum():,} | harmonizada {t2["sem_cnpj_na_harmonizada"].sum():,} (devem ser iguais)')
salvar(t2, '02_sem_cnpj_por_ano')

shape: (15, 5)
┌──────┬────────────────┬─────────────────────────────┬────────────────────┬─────────────────────────┐
│ ano  ┆ linhas_arquivo ┆ sem_cnpj_comprador_no_bruto ┆ dos_quais_sem_nome ┆ sem_cnpj_na_harmonizada │
│ ---  ┆ ---            ┆ ---                         ┆ ---                ┆ ---                     │
│ i64  ┆ i64            ┆ i64                         ┆ i64                ┆ u32                     │
╞══════╪════════════════╪═════════════════════════════╪════════════════════╪═════════════════════════╡
│ 2009 ┆ 13463          ┆ 27                          ┆ 0                  ┆ 27                      │
│ 2010 ┆ 19066          ┆ 38                          ┆ 12                 ┆ 38                      │
│ 2011 ┆ 13611          ┆ 99                          ┆ 4                  ┆ 99                      │
│ 2012 ┆ 4875           ┆ 0                           ┆ 0                  ┆ 0                       │
│ 2013 ┆ 3219           ┆ 9                           ┆ 0 

## 3. Recuperação de UF e município pelo CNPJ do comprador
**Decisão (D15 e D26).** Campos vazios são preenchidos na rotina 03, em dois passos: (1) pelo mesmo CNPJ em outros anos do BPS, só quando ele aparece sempre no mesmo local; (2) o que continuar vazio, pelo cadastro de Estabelecimentos da Receita Federal (dados abertos do CNPJ, dez/2023). Só campos vazios são preenchidos.

**Testes (gravados pela rotina 03).** `03a`: validação "deixa um ano de fora" do passo 1, por ano. `03b`: comparação da UF da Receita com a UF informada pela própria fonte, antes de aplicar o passo 2. `03c`: compras preenchidas pela Receita.

**Sobre o limite de 99,9% (D37).** É um limite do próprio trabalho, não vem de norma nem de autor: uma trava de segurança, que tolera até 1 erro em 1.000. Vale igual nos dois passos, os únicos lugares onde a UF é recuperada (a rotina 07 só confere, D38): cada passo só é aplicado se a concordância da UF for de pelo menos 99,9%. Os testes deram 100%. Aqui: os três registros e a cobertura final.

In [7]:
for nome in ['03a_validacao_uf_cnpj', '03b_validacao_uf_receita', '03c_preenchidos_receita']:
    arq = SAIDA / f'{nome}.csv'
    assert arq.exists(), f'{arq.name} não encontrado: rode a rotina 03 antes'
    print(f'\n{nome} (gravado pela rotina 03):'); print(pl.read_csv(arq, infer_schema_length=0))

h3 = ler(P_HARM, ['uf', 'municipio', 'local_recuperado', 'origem_local'])
t3 = (h3.group_by('arquivo').agg(pl.len().alias('linhas'),
        ((pl.col('uf').is_not_null() & (pl.col('uf') != '')).mean() * 100).round(1).alias('pct_uf'),
        ((pl.col('municipio').is_not_null() & (pl.col('municipio') != '')).mean() * 100).round(1).alias('pct_municipio'),
        pl.col('local_recuperado').cast(pl.Boolean).sum().alias('recuperados'),
        (pl.col('origem_local') == 'BPS, mesmo CNPJ em outros anos').sum().alias('recuperados_bps'),
        (pl.col('origem_local') == 'Receita Federal, Estabelecimentos dez/2023').sum().alias('recuperados_receita'))
        .rename({'arquivo': 'ano'}).sort('ano'))
print('\nCobertura final:'); print(t3); salvar(t3, '03_cobertura_uf_municipio')



03a_validacao_uf_cnpj (gravado pela rotina 03):
shape: (16, 8)
┌──────────┬────────────────────┬────────────┬────────────────────────┬────────┬────────────────────┬──────────────────────┬───────────────────────────────────────────────────┐
│ arquivo  ┆ registros_testados ┆ acertos_uf ┆ acertos_uf_e_municipio ┆ pct_uf ┆ pct_uf_e_municipio ┆ limite_minimo_uf_pct ┆ origem_do_limite                                  │
│ ---      ┆ ---                ┆ ---        ┆ ---                    ┆ ---    ┆ ---                ┆ ---                  ┆ ---                                               │
│ str      ┆ str                ┆ str        ┆ str                    ┆ str    ┆ str                ┆ str                  ┆ str                                               │
╞══════════╪════════════════════╪════════════╪════════════════════════╪════════╪════════════════════╪══════════════════════╪═══════════════════════════════════════════════════╡
│ BPS_2009 ┆ 12791              ┆ 12791      ┆ 1279

## 4. Valores implausíveis (erros de digitação da fonte)
**Decisão.** O valor total em reais não é apresentado como resultado; o HHI por valor exclui registros com
preço mais de 20× acima/abaixo da mediana do item no ano, ou quantidade mais de 200× acima (itens com ≥ 3 compras).
**Teste.** (a) peso dos maiores registros no valor; (b) os registros mais extremos; (c) sensibilidade a três réguas; (d) alcance da régua: quanto do valor está em item-ano com menos de 3 compras, onde ela não tem com o que comparar (D27).

In [8]:
v = pub.filter((pl.col('valor_total') > 0) & (pl.col('quantidade') > 0)).with_columns(
        pl.col('preco_real').median().over(['item', 'ano']).alias('p_med'),   # D16: régua sobre o preço real
        pl.col('quantidade').median().over(['item', 'ano']).alias('q_med'),
        pl.len().over(['item', 'ano']).alias('n_item'))
VT = v['valor_total'].sum()
top = v.sort('valor_total', descending=True)
print(f'Valor total registrado 2009-2023        : R$ {VT/1e9:,.1f} bilhoes')
print(f'  os 20 maiores registros somam          : R$ {top.head(20)["valor_total"].sum()/1e9:,.1f} bilhoes '
      f'= {top.head(20)["valor_total"].sum()/VT:.1%}')
ext = top.head(20).select('ano', pl.col('desc_item').str.slice(0, 50).alias('item'), 'unidade_chave', 'preco_real',
                          pl.col('p_med').alias('preco_mediano_item'), 'quantidade', pl.col('q_med').alias('qtd_mediana_item'),
                          (pl.col('valor_total') / 1e9).round(2).alias('valor_bi'), pl.col('instituicao').str.slice(0, 40))
print('\nOs 20 registros de maior valor:'); print(ext)
salvar(ext, '04a_20_maiores_registros')
linhas = []
for k in [10, 20, 50]:
    f = (((pl.col('preco_real') / pl.col('p_med') > k) | (pl.col('preco_real') / pl.col('p_med') < 1 / k)
          | (pl.col('quantidade') / pl.col('q_med') > k * 10)) & (pl.col('n_item') >= MIN_REG_REGUA))
    s = v.filter(f)
    linhas.append({'regua': f'{k}x preco / {k*10}x quantidade', 'registros_marcados': s.height,
                   'pct_registros': round(s.height / v.height * 100, 1), 'pct_do_valor': round(s['valor_total'].sum() / VT * 100, 1),
                   'valor_restante_bi': round((VT - s['valor_total'].sum()) / 1e9, 1)})
t4 = pl.DataFrame(linhas); print('\nSensibilidade a regua:'); print(t4); salvar(t4, '04b_sensibilidade_regua')
n20 = t4.filter(pl.col('regua').str.starts_with(f'{REGUA_PRECO}x'))['registros_marcados'][0]
print(f'Conferência: a régua de {REGUA_PRECO}x recalculada aqui marca {n20:,} compras; a rotina 04B marcou {int(pub["valor_implausivel"].sum()):,}')

# (d) alcance da régua: parte do valor (sem implausíveis, em reais de dez/2023) que está em item-ano com menos
#     de 3 compras, onde a régua não tem com o que comparar (lastro da D27)
cnpj4 = pl.col('cnpj_fornecedor').cast(pl.Utf8).str.strip_chars()
b4 = (pub.filter(cnpj4.is_not_null() & (cnpj4.str.len_chars() > 0) & (cnpj4 != '00000000000000')
                 & (pl.col('quantidade') > 0) & (pl.col('valor_total') > 0))
         .with_columns(pl.col('preco_real').median().over(['item', 'ano']).alias('pm'),
                       pl.col('quantidade').median().over(['item', 'ano']).alias('qm'),
                       pl.len().over(['item', 'ano']).alias('n_ia'),
                       (pl.col('valor_total') * pl.col('fator_deflacao')).alias('valor_real'))
         .filter(~((((pl.col('preco_real') / pl.col('pm') > REGUA_PRECO) | (pl.col('preco_real') / pl.col('pm') < 1 / REGUA_PRECO)
                     | (pl.col('quantidade') / pl.col('qm') > REGUA_QTD)) & (pl.col('n_ia') >= MIN_REG_REGUA)))))
t4c = (b4.group_by('ano').agg(pl.len().alias('compras'), (pl.col('n_ia') < MIN_REG_REGUA).sum().alias('compras_fora_do_alcance'),
                              pl.col('valor_real').sum().alias('v'), pl.col('valor_real').filter(pl.col('n_ia') < MIN_REG_REGUA).sum().alias('v3'))
         .sort('ano').with_columns((pl.col('v3') / pl.col('v') * 100).round(1).alias('pct_valor_fora_do_alcance')).drop('v', 'v3'))
tot = b4.select((pl.col('valor_real').filter(pl.col('n_ia') < MIN_REG_REGUA).sum() / pl.col('valor_real').sum() * 100).round(1)).item()
t4c = pl.concat([t4c.with_columns(pl.col('ano').cast(pl.Utf8)),
                 pl.DataFrame({'ano': ['TOTAL'], 'compras': [b4.height], 'compras_fora_do_alcance': [int((b4['n_ia'] < MIN_REG_REGUA).sum())],
                               'pct_valor_fora_do_alcance': [tot]})], how='vertical_relaxed')
print('\n(d) Alcance da régua: valor em item-ano com menos de 3 compras'); print(t4c); salvar(t4c, '04c_alcance_da_regua')


Valor total registrado 2009-2023        : R$ 232.0 bilhoes
  os 20 maiores registros somam          : R$ 154.3 bilhoes = 66.5%

Os 20 registros de maior valor:
shape: (20, 9)
┌──────┬────────────────────────────────────────────────────┬───────────────┬───────────────┬────────────────────┬──────────────┬──────────────────┬──────────┬──────────────────────────────────────────┐
│ ano  ┆ item                                               ┆ unidade_chave ┆ preco_real    ┆ preco_mediano_item ┆ quantidade   ┆ qtd_mediana_item ┆ valor_bi ┆ instituicao                              │
│ ---  ┆ ---                                                ┆ ---           ┆ ---           ┆ ---                ┆ ---          ┆ ---              ┆ ---      ┆ ---                                      │
│ i32  ┆ str                                                ┆ str           ┆ f64           ┆ f64                ┆ f64          ┆ f64              ┆ f64      ┆ str                                      │
╞══════╪═════

## 5 e 6. Concentração: duas medidas e o mínimo de 7 compras
**Decisões.** (i) HHI por número de compras e por valor (régua 20×); (ii) só se classifica a combinação
item-ano com pelo menos 7 compras. **Testes.** Prova matemática e empírica do 7; sensibilidade ao mínimo e à régua.

In [9]:
cnpjf = pl.col('cnpj_fornecedor').cast(pl.Utf8).str.strip_chars()
hb = v.filter(cnpjf.is_not_null() & (cnpjf.str.len_chars() > 0) & (cnpjf != '00000000000000'))
def flag(k):   # k = régua de preço; a de quantidade é 10 vezes k, como na adotada (REGUA_QTD / REGUA_PRECO = 200 / 20)
    return ((((pl.col('preco_real') / pl.col('p_med') > k) | (pl.col('preco_real') / pl.col('p_med') < 1 / k)
             | (pl.col('quantidade') / pl.col('q_med') > k * 10)) & (pl.col('n_item') >= MIN_REG_REGUA)))
def hhi(df, peso, chave='item'):
    """chave='item' (CATMAT + unidade, D17) é a medida publicada; chave='cod_catmat' é o teste de robustez da D17 (6d)."""
    w = pl.len().cast(pl.Float64) if peso == 'compras' else pl.col('valor_real').sum()   # valor em reais de dez/2023 (D20)
    g = (df.group_by(['ano', chave, 'cnpj_fornecedor']).agg(w.alias('w'), pl.len().alias('r'))
           .with_columns((pl.col('w') / pl.col('w').sum().over(['ano', chave]) * 100).alias('s')))
    return g.group_by(['ano', chave]).agg((pl.col('s') ** 2).sum().round(6).alias('hhi'), pl.col('r').sum().alias('n'))   # 6 casas: ver rotina 06
hc = hhi(hb, 'compras')
hv = {k: hhi(hb.filter(~flag(k)), 'valor') for k in [10, 20, 50]}
assert hc.height == pl.read_parquet(BASE / 'Base HHI - Publico' / 'hhi_concentracao.parquet').height, 'combinações diferentes da rotina 06'

print('6a. PROVA DO MINIMO DE 7 — menor HHI possivel com n compras = 10.000 / n')
t6a = (hc.with_columns(pl.when(pl.col('n') >= 10).then(10).otherwise(pl.col('n')).alias('nc')).group_by('nc')
         .agg(pl.len().alias('combinacoes'), pl.col('hhi').min().round(0).alias('menor_hhi_compras'),
              (pl.col('hhi') < LIM_NAO).sum().alias('nao_concentradas_compras'))
         .join(hv[20].with_columns(pl.when(pl.col('n') >= 10).then(10).otherwise(pl.col('n')).alias('nc')).group_by('nc')
               .agg(pl.col('hhi').min().round(0).alias('menor_hhi_valor'), (pl.col('hhi') < LIM_NAO).sum().alias('nao_concentradas_valor')),
               on='nc', how='left')
         .with_columns((10000 / pl.col('nc')).round(0).alias('limite_teorico_10000_n'),
                       pl.when(pl.col('nc') == 10).then(pl.lit('10 ou mais')).otherwise(pl.col('nc').cast(pl.Utf8)).alias('compras'))
         .sort('nc').select('compras', 'combinacoes', 'limite_teorico_10000_n', 'menor_hhi_compras', 'menor_hhi_valor',
                            'nao_concentradas_compras', 'nao_concentradas_valor'))
print(t6a)
print('Leitura: nenhuma combinacao fica abaixo do limite teorico; ate 6 compras nenhuma e "nao concentrada";')
print('a primeira classificacao "nao concentrado" aparece com 7 compras (10.000/7 = 1.429 < 1.500, faixa do CADE).')
salvar(t6a, '06a_prova_minimo_7')

print('\n6b. SENSIBILIDADE AO MINIMO DE COMPRAS')
linhas = []
for m in [1, 5, 7, 10, 30]:
    c = hc.filter(pl.col('n') >= m); vv = hv[20].filter(pl.col('n') >= m)
    linhas.append({'minimo_compras': m, 'combinacoes': c.height, 'pct_combinacoes': round(c.height / hc.height * 100, 1),
                   'pct_compras_cobertas': round(c['n'].sum() / hb.height * 100, 1),
                   'alta_por_compras_pct': round((c['hhi'] > LIM_ALTO).mean() * 100, 1),
                   'nao_conc_por_compras_pct': round((c['hhi'] < LIM_NAO).mean() * 100, 1),
                   'alta_por_valor_pct': round((vv['hhi'] > LIM_ALTO).mean() * 100, 1),
                   'nao_conc_por_valor_pct': round((vv['hhi'] < LIM_NAO).mean() * 100, 1)})
t6b = pl.DataFrame(linhas); print(t6b); salvar(t6b, '06b_sensibilidade_minimo')

print(f'\n5. SENSIBILIDADE DO HHI POR VALOR A REGUA (combinacoes com >= {MIN_COMPRAS} compras)')
linhas = []
for k, h_ in hv.items():
    c = h_.filter(pl.col('n') >= MIN_COMPRAS)
    linhas.append({'regua': f'{k}x', 'combinacoes_classificadas': c.height, 'hhi_mediano': round(c['hhi'].median()),
                   'alta_pct': round((c['hhi'] > LIM_ALTO).mean() * 100, 1), 'moderada_pct': round(((c['hhi'] >= LIM_NAO) & (c['hhi'] <= LIM_ALTO)).mean() * 100, 1),
                   'nao_conc_pct': round((c['hhi'] < LIM_NAO).mean() * 100, 1)})
t5 = pl.DataFrame(linhas); print(t5); salvar(t5, '05_sensibilidade_regua_hhi_valor')

print('\n6c. FAIXAS DO NÚMERO DE FORNECEDORES (D33) — demonstração: com n fornecedores de fatias iguais, HHI = 10.000 / n')
t6c = (pl.DataFrame({'fornecedores': list(range(1, 11))})
         .with_columns((10000 / pl.col('fornecedores')).round(0).alias('hhi_fatias_iguais'))
         .with_columns(pl.when(pl.col('hhi_fatias_iguais') > LIM_ALTO).then(pl.lit('Altamente concentrado (> 2.500)'))
                         .when(pl.col('hhi_fatias_iguais') >= LIM_NAO).then(pl.lit('Moderadamente concentrado (1.500 a 2.500)'))
                         .otherwise(pl.lit('Não concentrado (< 1.500)')).alias('classe_cade'),
                       pl.when(pl.col('fornecedores') == 1).then(pl.lit('1 fornecedor'))
                         .when(pl.col('fornecedores') <= 3).then(pl.lit('2 a 3'))
                         .when(pl.col('fornecedores') <= 6).then(pl.lit('4 a 6')).otherwise(pl.lit('7 ou mais')).alias('faixa_d33')))
print(t6c)
print('Leitura: 2 e 3 fornecedores caem acima de 2.500; 4 a 6 entre 1.500 e 2.500; 7 ou mais abaixo de 1.500 (CADE, 2016).')
salvar(t6c, '06c_faixas_fornecedores_d33')

print(f'\n6d. CONCENTRAÇÃO POR CATMAT (TESTE DE ROBUSTEZ DA D17): item = CATMAT + unidade (publicado) x só CATMAT')
linhas = []
for rot, ch in [('CATMAT + unidade (publicado, D17)', 'item'), ('só CATMAT (robustez)', 'cod_catmat')]:
    cc_ = hhi(hb, 'compras', ch).filter(pl.col('n') >= MIN_COMPRAS)
    vv_ = hhi(hb.filter(~flag(REGUA_PRECO)), 'valor', ch).filter(pl.col('n') >= MIN_COMPRAS)
    linhas.append({'item': rot, 'combinacoes': hhi(hb, 'compras', ch).height, 'classificadas': cc_.height,
                   'alta_compras_pct': round((cc_['hhi'] > LIM_ALTO).mean() * 100, 1), 'moderada_compras_pct': round(((cc_['hhi'] >= LIM_NAO) & (cc_['hhi'] <= LIM_ALTO)).mean() * 100, 1),
                   'nao_conc_compras_pct': round((cc_['hhi'] < LIM_NAO).mean() * 100, 1), 'hhi_mediano_compras': round(cc_['hhi'].median()),
                   'alta_valor_pct': round((vv_['hhi'] > LIM_ALTO).mean() * 100, 1), 'nao_conc_valor_pct': round((vv_['hhi'] < LIM_NAO).mean() * 100, 1)})
t6d = pl.DataFrame(linhas); print(t6d); salvar(t6d, '06d_robustez_item_catmat_d17')

6a. PROVA DO MINIMO DE 7 — menor HHI possivel com n compras = 10.000 / n
shape: (10, 7)
┌────────────┬─────────────┬────────────────────────┬───────────────────┬─────────────────┬──────────────────────────┬────────────────────────┐
│ compras    ┆ combinacoes ┆ limite_teorico_10000_n ┆ menor_hhi_compras ┆ menor_hhi_valor ┆ nao_concentradas_compras ┆ nao_concentradas_valor │
│ ---        ┆ ---         ┆ ---                    ┆ ---               ┆ ---             ┆ ---                      ┆ ---                    │
│ str        ┆ u32         ┆ f64                    ┆ f64               ┆ f64             ┆ u32                      ┆ u32                    │
╞════════════╪═════════════╪════════════════════════╪═══════════════════╪═════════════════╪══════════════════════════╪════════════════════════╡
│ 1          ┆ 28350       ┆ 10000.0                ┆ 10000.0           ┆ 10000.0         ┆ 0                        ┆ 0                      │
│ 2          ┆ 12374       ┆ 5000.0             

## 7. Índice de preço real — sensibilidade e o início da série
**Decisão.** Índice encadeado de itens pareados (CATMAT + unidade), 2009–2023, com ressalvas.
**Testes.** Sensibilidade a preços extremos e à definição de unidade; decomposição de cada elo pelo número
de compras do item, que mostra a fragilidade de 2011–2014.
**Atualização 25/10/2026 (D45, complemento).** O índice publicado passou a exigir 3 compras válidas por item-ano. O 07a ganhou a coluna `indice_2023_min3` (os mesmos cenários com a regra nova); o 07c fica como referência da versão sem mínimo, e o 07c2 repete a decomposição com o índice novo.


In [10]:
def indice(d, min3=False):
    d = d.filter(pl.col('unidade_chave').is_not_null() & (pl.col('unidade_chave').str.len_chars() > 0))
    if min3:   # D45 (compl. 25/10/2026): só item-ano com 3 compras ou mais, contadas DENTRO do cenário (depois da régua ou da nova unidade)
        d = d.filter(pl.len().over(['item', 'ano']) >= 3)
    med = d.group_by(['ano', 'item']).agg(pl.col('preco_real').median().alias('p'))
    idx = 100.0
    for a in range(2009, 2023):
        j = med.filter(pl.col('ano') == a).join(med.filter(pl.col('ano') == a + 1), on='item')
        idx *= float((j['p_right'] / j['p']).median())
    return round(idx, 1)
base7 = pub.with_columns(pl.col('preco_real').median().over(['item', 'ano']).alias('p_med'),
                         pl.len().over(['item', 'ano']).alias('n_item'))
linhas = [{'cenario': 'indice publicado (rotina 05; sem implausíveis, D16)', 'indice_2023': indice(pub_val), 'indice_2023_min3': indice(pub_val, True)},
          {'cenario': 'com as compras implausíveis (antes da D16)', 'indice_2023': indice(pub), 'indice_2023_min3': indice(pub, True)}]
for k in [10, 20, 50]:
    f = (((pl.col('preco_real') / pl.col('p_med') > k) | (pl.col('preco_real') / pl.col('p_med') < 1 / k)) & (pl.col('n_item') >= MIN_REG_REGUA))
    linhas.append({'cenario': f'sem precos {k}x fora da mediana', 'indice_2023': indice(base7.filter(~f)), 'indice_2023_min3': indice(base7.filter(~f), True)})
# desde out/2026 a rotina 03 retira a marca GENERICO da unidade (regra publicada); aqui, o cenario inverso:
com_gen = pub_val.with_columns(pl.when(pl.col('unidade_marca_generico').fill_null(False))
                             .then(pl.col('unidade_chave') + 'GENERICO').otherwise(pl.col('unidade_chave')).alias('unidade_chave'))
com_gen = com_gen.with_columns((pl.col('cod_catmat') + '|' + pl.col('unidade_chave').fill_null('')).alias('item'))
linhas.append({'cenario': 'unidade COM a marca GENERICO (regra anterior)', 'indice_2023': indice(com_gen), 'indice_2023_min3': indice(com_gen, True)})
emb = pub_val.with_columns(pl.col('unidade_chave').str.replace_all(r'[\d,\.]+(ML|MG|G|L|MCG|UI|CM|MM|M)?', ''))
emb = emb.with_columns((pl.col('cod_catmat') + '|' + pl.col('unidade_chave').fill_null('')).alias('item'))
linhas.append({'cenario': 'unidade so pela embalagem (sem medida)', 'indice_2023': indice(emb), 'indice_2023_min3': indice(emb, True)})
t7a = pl.DataFrame(linhas)   # indice_2023: sem mínimo (versão anterior); indice_2023_min3: 3 compras ou mais (D45)
print(t7a); salvar(t7a, '07a_sensibilidade_indice')

print('\nMarca GENERICO na unidade e embalagem sem medida, por ano (% dos registros):')
EMB = ['FRASCO', 'AMPOLA', 'FRASCOAMPOLA', 'BISNAGA', 'TUBO', 'BOLSA', 'ENVELOPE', 'SACHE', 'POTE']
t7b = (pub.group_by('ano').agg((pl.col('unidade_marca_generico').fill_null(False).mean() * 100).round(1).alias('pct_generico_na_unidade'),
                               (pl.col('unidade_chave').is_in(EMB).mean() * 100).round(1).alias('pct_embalagem_sem_medida'))
          .sort('ano'))
print(t7b); salvar(t7b, '07b_formato_unidade_por_ano')

print('\nDecomposicao de cada elo pelo numero de compras do item (menor dos dois anos):')
d = pub_val.filter(pl.col('unidade_chave').is_not_null() & (pl.col('unidade_chave').str.len_chars() > 0))
med = d.group_by(['ano', 'item']).agg(pl.col('preco_real').median().alias('p'), pl.len().alias('n'))
linhas = []
for a in range(2009, 2023):
    j = (med.filter(pl.col('ano') == a).join(med.filter(pl.col('ano') == a + 1), on='item')
           .with_columns((pl.col('p_right') / pl.col('p')).alias('r'), pl.min_horizontal('n', 'n_right').alias('nmin')))
    lin = {'elo': f'{a}-{a+1}', 'pares': j.height}
    for rot, f in [('n1', pl.col('nmin') == 1), ('n2a4', pl.col('nmin').is_between(2, 4)), ('n5mais', pl.col('nmin') >= 5)]:
        s = j.filter(f)
        lin[f'itens_{rot}'] = s.height
        lin[f'var_{rot}_pct'] = round((float(s['r'].median()) - 1) * 100, 1) if s.height else None
    linhas.append(lin)
t7c = pl.DataFrame(linhas); print(t7c); salvar(t7c, '07c_decomposicao_elos')
for rot in ['n1', 'n2a4', 'n5mais']:
    idx = 100.0; idx14 = 100.0
    for r in t7c.iter_rows(named=True):
        if r[f'var_{rot}_pct'] is not None:
            idx *= 1 + r[f'var_{rot}_pct'] / 100
            if int(r['elo'][:4]) >= 2014: idx14 *= 1 + r[f'var_{rot}_pct'] / 100
    print(f'  grupo {rot:<7}: 2009-2023 = {idx:5.1f} | 2014-2023 = {idx14:5.1f}')
print('Leitura: de 2014 em diante todos os grupos indicam queda; antes de 2014 o grupo com 5+ compras')
print('apoia-se em poucas dezenas de itens -> magnitude do periodo inicial e fragil.')

shape: (7, 3)
┌─────────────────────────────────────────────────────┬─────────────┬──────────────────┐
│ cenario                                             ┆ indice_2023 ┆ indice_2023_min3 │
│ ---                                                 ┆ ---         ┆ ---              │
│ str                                                 ┆ f64         ┆ f64              │
╞═════════════════════════════════════════════════════╪═════════════╪══════════════════╡
│ indice publicado (rotina 05; sem implausíveis, D16) ┆ 77.1        ┆ 86.9             │
│ com as compras implausíveis (antes da D16)          ┆ 76.7        ┆ 87.3             │
│ sem precos 10x fora da mediana                      ┆ 76.7        ┆ 87.5             │
│ sem precos 20x fora da mediana                      ┆ 76.6        ┆ 87.8             │
│ sem precos 50x fora da mediana                      ┆ 76.6        ┆ 87.3             │
│ unidade COM a marca GENERICO (regra anterior)       ┆ 77.2        ┆ 85.5             │
│ unida

In [11]:
# 07c2 (D45, compl. 25/10/2026): a mesma decomposição do 07c, com o índice novo (3 compras válidas ou mais por item-ano).
# Com o mínimo, as faixas "1 compra" e "2 a 4" deixam de existir; ficam "3 a 4" e "5 ou mais". O 07c fica como referência da versão sem mínimo.
print('Decomposicao de cada elo pelo numero de compras do item, com 3 compras validas ou mais (menor dos dois anos):')
d3 = (pub_val.filter(pl.col('unidade_chave').is_not_null() & (pl.col('unidade_chave').str.len_chars() > 0))
             .filter(pl.len().over(['item', 'ano']) >= 3))
med3 = d3.group_by(['ano', 'item']).agg(pl.col('preco_real').median().alias('p'), pl.len().alias('n'))
linhas = []
for a in range(2009, 2023):
    j = (med3.filter(pl.col('ano') == a).join(med3.filter(pl.col('ano') == a + 1), on='item')
             .with_columns((pl.col('p_right') / pl.col('p')).alias('r'), pl.min_horizontal('n', 'n_right').alias('nmin')))
    lin = {'elo': f'{a}-{a+1}', 'pares': j.height}
    for rot, f in [('n3a4', pl.col('nmin').is_between(3, 4)), ('n5mais', pl.col('nmin') >= 5)]:
        s_ = j.filter(f)
        lin[f'itens_{rot}'] = s_.height
        lin[f'var_{rot}_pct'] = round((float(s_['r'].median()) - 1) * 100, 1) if s_.height else None
    linhas.append(lin)
t7c2 = pl.DataFrame(linhas); print(t7c2); salvar(t7c2, '07c2_decomposicao_elos_min3')
for rot in ['n3a4', 'n5mais']:
    idx = 100.0; idx14 = 100.0
    for r in t7c2.iter_rows(named=True):
        if r[f'var_{rot}_pct'] is not None:
            idx *= 1 + r[f'var_{rot}_pct'] / 100
            if int(r['elo'][:4]) >= 2014: idx14 *= 1 + r[f'var_{rot}_pct'] / 100
    print(f'  grupo {rot:<7}: 2009-2023 = {idx:5.1f} | 2014-2023 = {idx14:5.1f}')


Decomposicao de cada elo pelo numero de compras do item, com 3 compras validas ou mais (menor dos dois anos):
shape: (14, 6)
┌───────────┬───────┬────────────┬──────────────┬──────────────┬────────────────┐
│ elo       ┆ pares ┆ itens_n3a4 ┆ var_n3a4_pct ┆ itens_n5mais ┆ var_n5mais_pct │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---          ┆ ---            │
│ str       ┆ i64   ┆ i64        ┆ f64          ┆ i64          ┆ f64            │
╞═══════════╪═══════╪════════════╪══════════════╪══════════════╪════════════════╡
│ 2009-2010 ┆ 578   ┆ 249        ┆ -1.7         ┆ 329          ┆ -5.4           │
│ 2010-2011 ┆ 697   ┆ 277        ┆ -3.1         ┆ 420          ┆ -5.0           │
│ 2011-2012 ┆ 328   ┆ 199        ┆ -2.6         ┆ 129          ┆ 4.3            │
│ 2012-2013 ┆ 161   ┆ 116        ┆ -3.3         ┆ 45           ┆ 8.6            │
│ 2013-2014 ┆ 238   ┆ 171        ┆ 8.9          ┆ 67           ┆ 14.0           │
│ 2014-2015 ┆ 1257  ┆ 387        ┆ -1.6         ┆ 870  

## 8. Escala — item definido como CATMAT + unidade e mínimo de 3 compras (D17, D53)
**Decisões.** A rotina 08 usa a mesma definição de item do índice de preço (CATMAT + unidade, D17) e só item-ano com 3 compras
válidas ou mais (D53, o mesmo mínimo da D45): com 1 compra, a compra seria comparada consigo mesma (desvio zero); com 2, a
mediana é a média das duas, e a régua de valores implausíveis não confere nenhuma delas.
**Teste.** Desvio de preço por quintil de lote em três versões: só CATMAT (versão anterior), CATMAT + unidade com todos os
item-ano (antes da D53) e a publicada (CATMAT + unidade, 3 compras ou mais, colunas da rotina 04B). As duas primeiras usam a
mediana dos logaritmos, por isso diferem da publicada em até 0,1 ponto mesmo sem a D53; a terceira tem de ser igual à rotina 08.

In [12]:
e = pub_val.filter((pl.col('quantidade') > 0) & pl.col('unidade_chave').is_not_null() & (pl.col('unidade_chave').str.len_chars() > 0)
               ).with_columns(pl.col('preco_real').log().alias('lp'), pl.col('quantidade').log().alias('lq'))
linhas = []
for rot, chave in [('so CATMAT (versao anterior)', ['cod_catmat', 'ano']), ('CATMAT + unidade, todos os item-ano (antes da D53)', ['cod_catmat', 'unidade_chave', 'ano'])]:
    x = (e.with_columns((pl.col('lp') - pl.col('lp').median().over(chave)).alias('dp'),
                        (pl.col('lq') - pl.col('lq').median().over(chave)).alias('dq'))
          .with_columns(pl.col('dq').qcut(5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'], allow_duplicates=True).alias('q')))
    t = x.group_by('q').agg(((pl.col('dp').median().exp() - 1) * 100).round(1).alias('d')).sort('q')
    linhas.append({'chave': rot, **dict(zip(t['q'].cast(pl.Utf8), t['d']))})
# publicada (D53): as mesmas colunas e o mesmo filtro da rotina 08
pubq = (pub_val.filter((pl.col('quantidade') > 0) & (pl.col('compras_validas_item_ano') >= MIN_MEDIANA))
               .group_by('faixa_lote').agg(((pl.col('preco_rel_mediana').log().median().exp() - 1) * 100).round(1).alias('d')).sort('faixa_lote'))
linhas.append({'chave': f'CATMAT + unidade, item-ano com {MIN_MEDIANA} compras ou mais (publicada, D53)', **dict(zip(pubq['faixa_lote'], pubq['d']))})
u = pub.group_by(['cod_catmat', 'ano']).agg(pl.col('unidade_chave').n_unique().alias('k'), pl.len().alias('n'))
t8 = pl.DataFrame(linhas); print(t8); salvar(t8, '08_escala_chave_item')
q08 = pl.read_csv(BASE / 'Base Escala - Publico' / 'escala_quintis.csv')
assert q08['desvio_preco_pct'].to_list() == pubq['d'].to_list(), 'a versão publicada difere da rotina 08: rode a 08 antes'
fora53 = pub_val.filter((pl.col('quantidade') > 0) & (pl.col('compras_validas_item_ano') < MIN_MEDIANA))
print(f'Fora da escala pela D53: {fora53.height:,} compras ({fora53.filter(pl.col("compras_validas_item_ano") == 1).height:,} em item-ano de 1 compra)')
print(f'Registros em CATMAT-ano com mais de uma unidade: {u.filter(pl.col("k") > 1)["n"].sum() / u["n"].sum():.1%}')

shape: (3, 6)
┌───────────────────────────────────────────────────────────────┬──────┬─────┬─────┬──────┬──────┐
│ chave                                                         ┆ Q1   ┆ Q2  ┆ Q3  ┆ Q4   ┆ Q5   │
│ ---                                                           ┆ ---  ┆ --- ┆ --- ┆ ---  ┆ ---  │
│ str                                                           ┆ f64  ┆ f64 ┆ f64 ┆ f64  ┆ f64  │
╞═══════════════════════════════════════════════════════════════╪══════╪═════╪═════╪══════╪══════╡
│ so CATMAT (versao anterior)                                   ┆ 16.4 ┆ 2.7 ┆ 0.0 ┆ -2.2 ┆ -8.8 │
│ CATMAT + unidade, todos os item-ano (antes da D53)            ┆ 14.8 ┆ 2.5 ┆ 0.0 ┆ -2.0 ┆ -8.3 │
│ CATMAT + unidade, item-ano com 3 compras ou mais (publicada,… ┆ 15.2 ┆ 2.7 ┆ 0.0 ┆ -2.2 ┆ -8.3 │
└───────────────────────────────────────────────────────────────┴──────┴─────┴─────┴──────┴──────┘
  -> Auditoria/08_escala_chave_item.csv
Fora da escala pela D53: 54,976 compras (28,374 em item

## 9. Índice territorial — estabilidade e a ressalva de Roraima
**Decisão.** Roraima permanece no índice, com ressalva: poucos compradores, um deles predominante.

**Mínimo por estado (D23).** Teste (b): para vários cortes, quantos estados entram e quais ficam de fora; a faixa de cortes que dá exatamente o mesmo resultado do 500 sustenta o número (arquivo 09b).

In [13]:
# D39: a contagem de UFs ignora compras sem UF (antes, o vazio contava como mais uma UF); mesma regra da rotina 07,
# que só conta UFs entre as compras com UF
_tem_uf = pl.col('uf').is_not_null() & (pl.col('uf') != '')
nac = pub_val.filter(_tem_uf).group_by(['ano', 'item']).agg(pl.col('preco_real').median().alias('pn'), pl.col('uf').n_unique().alias('nuf'))
r9 = (pub_val.filter(pl.col('uf').is_not_null() & (pl.col('uf') != '')).join(nac, on=['ano', 'item'])
         .filter(pl.col('nuf') >= 5).with_columns((pl.col('preco_real') / pl.col('pn')).alias('rel')))
def idx_uf(df, rot):
    return (df.group_by('uf').agg(pl.col('rel').median().round(3).alias(rot), pl.len().alias(f'reg_{rot}'))
              .filter(pl.col(f'reg_{rot}') >= 500))
t9 = (idx_uf(r9, 'total').join(idx_uf(r9.filter(pl.col('ano') <= 2016), '2009_16'), on='uf', how='left')
        .join(idx_uf(r9.filter(pl.col('ano') >= 2017), '2017_23'), on='uf', how='left'))
g = r9.join(t9.select('uf'), on='uf').group_by('uf', 'cnpj_instituicao').len()
t9 = t9.join(g.group_by('uf').agg(pl.len().alias('compradores'), (pl.col('len').max() / pl.col('len').sum() * 100).round(1).alias('pct_maior_comprador')),
             on='uf').sort('total')
print(t9)
tt = t9.drop_nulls(['2009_16', '2017_23'])
print(f'Estabilidade entre periodos: Spearman {spearmanr(tt["2009_16"], tt["2017_23"])[0]:.2f} ({tt.height} UFs)')
rr = r9.filter(pl.col('uf') == 'RR')
maior = rr.group_by('cnpj_instituicao').len().sort('len', descending=True)['cnpj_instituicao'][0]
print(f'Roraima: indice {rr["rel"].median():.3f} | sem o maior comprador: {rr.filter(pl.col("cnpj_instituicao") != maior)["rel"].median():.3f}')
print(rr.group_by('instituicao').agg(pl.len().alias('registros'), pl.col('rel').median().round(3).alias('indice'),
                                     pl.col('ano').min().alias('de'), pl.col('ano').max().alias('ate')).sort('registros', descending=True))
salvar(t9, '09_territorial_estabilidade')

shape: (23, 9)
┌─────┬───────┬───────────┬─────────┬─────────────┬─────────┬─────────────┬─────────────┬─────────────────────┐
│ uf  ┆ total ┆ reg_total ┆ 2009_16 ┆ reg_2009_16 ┆ 2017_23 ┆ reg_2017_23 ┆ compradores ┆ pct_maior_comprador │
│ --- ┆ ---   ┆ ---       ┆ ---     ┆ ---         ┆ ---     ┆ ---         ┆ ---         ┆ ---                 │
│ str ┆ f64   ┆ u32       ┆ f64     ┆ u32         ┆ f64     ┆ u32         ┆ u32         ┆ f64                 │
╞═════╪═══════╪═══════════╪═════════╪═════════════╪═════════╪═════════════╪═════════════╪═════════════════════╡
│ SC  ┆ 0.912 ┆ 32095     ┆ 0.935   ┆ 6479        ┆ 0.908   ┆ 25616       ┆ 86          ┆ 18.5                │
│ RS  ┆ 0.927 ┆ 26641     ┆ 0.935   ┆ 7710        ┆ 0.924   ┆ 18931       ┆ 79          ┆ 17.9                │
│ PR  ┆ 0.969 ┆ 134330    ┆ 0.979   ┆ 27594       ┆ 0.967   ┆ 106736      ┆ 307         ┆ 2.4                 │
│ SP  ┆ 0.971 ┆ 169766    ┆ 0.962   ┆ 61719       ┆ 0.977   ┆ 108047      ┆ 367         ┆

In [14]:
# 9b. Mínimo de registros por estado (D23): 500 compras ou mais no período 2009–2023.
# O corte só decide QUAIS estados entram; o índice de cada estado não depende do corte. O teste mostra, para vários
# cortes, quantos estados entram e quais ficam de fora, e a faixa de cortes que dá exatamente o mesmo resultado do 500.
CORTES_TESTADOS = [50, 100, 110, 200, 300, 400, 500, 600, 650, 700, 800, 1000]   # grade do teste (D23); o corte adotado é 500
CORTE_D23 = 500                                                                  # D23
reg_uf = r9.group_by('uf').agg(pl.len().alias('registros'), pl.col('rel').median().round(3).alias('indice')).sort('registros')
lin = []
for c in CORTES_TESTADOS:
    k = reg_uf.filter(pl.col('registros') >= c)
    lin.append({'corte_registros': c, 'estados': k.height,
                'estados_fora': ', '.join(reg_uf.filter(pl.col('registros') < c)['uf'].to_list()),
                'menor_indice': k['indice'].min(), 'maior_indice': k['indice'].max()})
t9b = pl.DataFrame(lin)
print('9b. MÍNIMO DE REGISTROS POR ESTADO — o que muda com o corte'); print(t9b)
abaixo, acima = reg_uf.filter(pl.col('registros') < CORTE_D23), reg_uf.filter(pl.col('registros') >= CORTE_D23)
de, ate = (abaixo['registros'].max() or 0) + 1, acima['registros'].min()
print(f'Qualquer corte de {de} a {ate} registros dá exatamente o mesmo conjunto de estados (e os mesmos índices) que o corte de {CORTE_D23}.')
print('Estados com menos de 1.000 registros no período:'); print(reg_uf.filter(pl.col('registros') < 1000))
salvar(t9b.with_columns(pl.lit(f'{de} a {ate}').alias('faixa_de_cortes_com_resultado_igual_ao_500')), '09b_sensibilidade_minimo_estado')


9b. MÍNIMO DE REGISTROS POR ESTADO — o que muda com o corte
shape: (12, 5)
┌─────────────────┬─────────┬────────────────────┬──────────────┬──────────────┐
│ corte_registros ┆ estados ┆ estados_fora       ┆ menor_indice ┆ maior_indice │
│ ---             ┆ ---     ┆ ---                ┆ ---          ┆ ---          │
│ i64             ┆ i64     ┆ str                ┆ f64          ┆ f64          │
╞═════════════════╪═════════╪════════════════════╪══════════════╪══════════════╡
│ 50              ┆ 24      ┆ AM, MA, DF         ┆ 0.912        ┆ 1.802        │
│ 100             ┆ 24      ┆ AM, MA, DF         ┆ 0.912        ┆ 1.802        │
│ 110             ┆ 23      ┆ AM, MA, DF, AP     ┆ 0.912        ┆ 1.802        │
│ 200             ┆ 23      ┆ AM, MA, DF, AP     ┆ 0.912        ┆ 1.802        │
│ 300             ┆ 23      ┆ AM, MA, DF, AP     ┆ 0.912        ┆ 1.802        │
│ 400             ┆ 23      ┆ AM, MA, DF, AP     ┆ 0.912        ┆ 1.802        │
│ 500             ┆ 23      ┆ AM, 

## 10. Representatividade da base ano a ano
O BPS é de registro voluntário: o volume reflete adesão ao sistema, não só o tamanho do mercado.

In [15]:
# D39: 'ufs' conta só UFs preenchidas (antes, compras sem UF contavam como mais uma UF)
t10 = (pub.group_by('ano').agg(pl.len().alias('registros'), pl.col('cnpj_instituicao').n_unique().alias('compradores'),
                               pl.col('item').n_unique().alias('itens'), pl.col('cod_catmat').n_unique().alias('codigos_catmat'), pl.col('cnpj_fornecedor').n_unique().alias('fornecedores'),
                               pl.col('uf').filter(pl.col('uf').is_not_null() & (pl.col('uf') != '')).n_unique().alias('ufs'), (pl.col('natureza_desc').str.contains('Munic').mean() * 100).round(1).alias('pct_municipal'))
          .sort('ano'))
print(t10)
por_uf = pub.group_by('uf').len().sort('len', descending=True)
print(f'Compradores distintos no periodo: {pub["cnpj_instituicao"].n_unique():,}')
print(f'SP + PR: {por_uf.filter(pl.col("uf").is_in(["SP", "PR"]))["len"].sum() / pub.height:.1%} dos registros')
salvar(t10, '10_representatividade_por_ano')

shape: (15, 8)
┌──────┬───────────┬─────────────┬───────┬────────────────┬──────────────┬─────┬───────────────┐
│ ano  ┆ registros ┆ compradores ┆ itens ┆ codigos_catmat ┆ fornecedores ┆ ufs ┆ pct_municipal │
│ ---  ┆ ---       ┆ ---         ┆ ---   ┆ ---            ┆ ---          ┆ --- ┆ ---           │
│ i32  ┆ u32       ┆ u32         ┆ u32   ┆ u32            ┆ u32          ┆ u32 ┆ f64           │
╞══════╪═══════════╪═════════════╪═══════╪════════════════╪══════════════╪═════╪═══════════════╡
│ 2009 ┆ 5105      ┆ 70          ┆ 1718  ┆ 1567           ┆ 327          ┆ 22  ┆ 35.3          │
│ 2010 ┆ 10063     ┆ 172         ┆ 2075  ┆ 1862           ┆ 440          ┆ 26  ┆ 52.8          │
│ 2011 ┆ 8299      ┆ 65          ┆ 2187  ┆ 1978           ┆ 401          ┆ 21  ┆ 60.1          │
│ 2012 ┆ 3214      ┆ 27          ┆ 1445  ┆ 1361           ┆ 251          ┆ 12  ┆ 44.6          │
│ 2013 ┆ 2169      ┆ 30          ┆ 1130  ┆ 1053           ┆ 222          ┆ 12  ┆ 51.1          │
│ 2014 ┆ 19384 

## 11. Capacidade dos dados — fabricante, classe do material, medicamentos x materiais

In [16]:
fb = ler(P_HARM, ['cnpj_fabricante', 'cnpj_fornecedor'])
cf = pl.col('cnpj_fabricante').cast(pl.Utf8).str.replace_all(r'\D', '')
t11a = (fb.with_columns(cf.alias('f'), pl.col('cnpj_fornecedor').cast(pl.Utf8).str.replace_all(r'\D', '').alias('v'))
          .with_columns((pl.col('f').str.len_chars() >= 11).fill_null(False).alias('ok'))
          .group_by('arquivo').agg((pl.col('ok').mean() * 100).round(1).alias('pct_cnpj_fabricante'),
                                   (((pl.col('f') == pl.col('v')) & pl.col('ok')).mean() * 100).round(1).alias('pct_fabricante_igual_fornecedor'))
          .rename({'arquivo': 'ano'}).sort('ano'))
print('Fabricante por ano:'); print(t11a); salvar(t11a, '11a_fabricante_por_ano')

# Fase 2 (D40, D41): a classe vem da base analítica — BPS 2018–2020 e, para os demais códigos, o catálogo oficial
t11b = (pub.group_by('ano').agg((pl.col('origem_classe') == 'BPS 2018-2020').sum().alias('classe_do_bps'),
                                (pl.col('origem_classe') != 'BPS 2018-2020').sum().alias('classe_do_catalogo'),
                                ((pl.col('item').filter(pl.col('classe_catmat') != '6505').n_unique())
                                 / pl.col('item').n_unique() * 100).round(1).alias('pct_itens_materiais'))
          .sort('ano'))
print(t11b); salvar(t11b, '11b_classe_e_materiais_por_ano')
print(f'Periodo inteiro: medicamentos (classe 6505) {pub.filter(pl.col("classe_catmat") == "6505")["item"].n_unique() / pub["item"].n_unique():.1%} '
      f'dos itens | {(pub["classe_catmat"] == "6505").mean():.1%} das compras')
print('Classe 6505 = medicamentos; demais classes de saúde = materiais, equipamentos e outros produtos para a saúde.')

Fabricante por ano:
shape: (15, 3)
┌──────┬─────────────────────┬─────────────────────────────────┐
│ ano  ┆ pct_cnpj_fabricante ┆ pct_fabricante_igual_fornecedor │
│ ---  ┆ ---                 ┆ ---                             │
│ i32  ┆ f64                 ┆ f64                             │
╞══════╪═════════════════════╪═════════════════════════════════╡
│ 2009 ┆ 87.5                ┆ 76.8                            │
│ 2010 ┆ 91.7                ┆ 79.8                            │
│ 2011 ┆ 89.4                ┆ 66.4                            │
│ 2012 ┆ 64.3                ┆ 8.4                             │
│ 2013 ┆ 81.8                ┆ 8.3                             │
│ 2014 ┆ 99.3                ┆ 6.0                             │
│ 2015 ┆ 99.9                ┆ 3.9                             │
│ 2016 ┆ 99.9                ┆ 4.0                             │
│ 2017 ┆ 100.0               ┆ 4.6                             │
│ 2018 ┆ 100.0               ┆ 3.3                     

## 12. Duplicatas exatas da fonte
**Decisão.** Uma linha é retirada quando é idêntica a outra em **todas** as colunas do arquivo de origem
(instituição, data da compra, data de inserção, item, fornecedor, fabricante, quantidade, preço etc.); a primeira
ocorrência fica (rotina 03, etapa 3.0). Linhas que diferem em qualquer coluna, inclusive só na data de inserção,
permanecem.
**Testes.** (a) contagem refeita aqui, sobre o CSV convertido, e conferida com a rotina 03; (b) quantas vezes as linhas
se repetem e em quais compradores; (c) distribuição de 2023 por mês da compra; (d) efeito da retirada nos indicadores,
reconstruindo a base original pela coluna `copias_retiradas`; (e) ressalva: linhas iguais exceto pela data de inserção.

In [17]:
# (a) contagem refeita a partir do CSV convertido e conferida com a memoria de calculo da rotina 03
dup03 = pl.read_csv(P_HARM / 'duplicatas_exatas_por_ano.csv')
linhas, brutos = [], {}
for a in ANOS:
    raw = pl.read_csv(P_CSV / f'BPS_{a}.csv', infer_schema_length=0, encoding='utf8-lossy')
    brutos[a] = raw
    ins = [c for c in raw.columns if 'inser' in c.lower()]
    k = raw.height - raw.unique().height
    k_sem_ins = raw.height - raw.unique(subset=[c for c in raw.columns if c not in ins]).height
    linhas.append({'ano': a, 'linhas_fonte': raw.height, 'duplicatas_exatas': k,
                   'pct_duplicatas': round(k / raw.height * 100, 2),
                   'iguais_exceto_data_insercao': k_sem_ins - k})
t12a = (pl.DataFrame(linhas).join(dup03.select('ano', pl.col('duplicatas_retiradas').alias('retiradas_rotina_03')), on='ano', how='left')
          .with_columns((pl.col('duplicatas_exatas') == pl.col('retiradas_rotina_03')).alias('confere')))
print('12a. DUPLICATAS EXATAS POR ANO (todas as colunas do arquivo de origem)')
print(t12a)
print(f"Total 2009-2023: {t12a['duplicatas_exatas'].sum():,} de {t12a['linhas_fonte'].sum():,} linhas "
      f"= {t12a['duplicatas_exatas'].sum() / t12a['linhas_fonte'].sum():.2%}")
assert t12a['confere'].all(), 'contagem diverge da rotina 03'
salvar(t12a, '12a_duplicatas_por_ano')

# (b) quantas vezes cada linha de 2023 se repete, e em quais compradores estao as copias
r23 = brutos[2023]
grp = r23.group_by(r23.columns).agg(pl.len().alias('vezes'))
t12b = (grp.with_columns(pl.col('vezes').cut([1, 2, 5, 10, 20], labels=['1 (sem copia)', '2', '3 a 5', '6 a 10', '11 a 20', '21 a 50']).alias('faixa'))
           .group_by('faixa').agg(pl.len().alias('linhas_distintas'), pl.col('vezes').sum().alias('linhas_no_arquivo'),
                                  pl.col('vezes').min().alias('_ordem'))
           .sort('_ordem').drop('_ordem').with_columns(pl.col('faixa').cast(pl.Utf8)))
print('\n12b. 2023 — QUANTAS VEZES CADA LINHA APARECE NO ARQUIVO')
print(t12b); print(f"maior repeticao: {grp['vezes'].max()} vezes")
salvar(t12b, '12b_repeticoes_2023')
t12c = (grp.filter(pl.col('vezes') > 1)
           .group_by('Nome Instituição', 'UF').agg((pl.col('vezes') - 1).sum().alias('copias_retiradas'),
                                                     pl.col('Inserção').min().alias('insercao_min'),
                                                     pl.col('Inserção').max().alias('insercao_max'))
           .sort(['copias_retiradas', 'Nome Instituição', 'UF'], descending=[True, False, False]))
print(f'\nCompradores com copias em 2023: {t12c.height} | os 10 maiores respondem por '
      f"{t12c.head(10)['copias_retiradas'].sum() / t12c['copias_retiradas'].sum():.1%} das copias")
print(t12c.head(10))
salvar(t12c, '12c_compradores_copias_2023')

# (c) 2023 por mes da compra: ha registros em todos os meses, mas o volume cai a partir de abril
#     (correcao de 12/10/2026: a leitura anterior, 'ano incompleto', estava errada; ver D2 no registro de decisoes)
t12d = (r23.with_columns(pl.col('Compra').str.slice(0, 7).alias('mes'),
                         (~pl.struct(pl.all()).is_first_distinct()).alias('copia'))
           .group_by('mes').agg(pl.len().alias('linhas_fonte'), pl.col('copia').sum().alias('copias'))
           .with_columns((pl.col('linhas_fonte') - pl.col('copias')).alias('linhas_apos'))
           .sort('mes'))
print('\n12d. 2023 POR MES DA COMPRA'); print(t12d)
print(f"jan-mar = {t12d.filter(pl.col('mes') <= '2023-03')['linhas_apos'].sum() / t12d['linhas_apos'].sum():.1%} "
      'das linhas apos a retirada (no arquivo bruto: '
      f"{t12d.filter(pl.col('mes') <= '2023-03')['linhas_fonte'].sum() / t12d['linhas_fonte'].sum():.1%})")
# comparacao com 2022 na base publica: o 1o trimestre de 2023 esta no nivel dos anos anteriores; de abril a dezembro, nao
pm = (pub.filter(pl.col('data_compra').dt.year().is_in([2022, 2023]))
         .group_by(pl.col('data_compra').dt.year().alias('ano'), (pl.col('data_compra').dt.month() >= 4).alias('abr_dez'))
         .agg(pl.len().alias('compras')))
v = {(a, d): n for a, d, n in pm.rows()}
print(f"base publica, jan-mar: 2022 = {v[(2022, False)]:,} | 2023 = {v[(2023, False)]:,}")
print(f"base publica, abr-dez: 2022 = {v[(2022, True)]:,} | 2023 = {v[(2023, True)]:,} "
      f"= {v[(2023, True)] / v[(2022, True)]:.0%} do volume de 2022")
print('Leitura: 2023 tem compras em todos os meses, mas de abril a dezembro o volume registrado e muito menor que nos anos '
      'anteriores. A fonte nao informa o motivo (a data de insercao mais recente no arquivo e fev/2025).')
salvar(t12d, '12d_2023_por_mes')

# (d) efeito nos indicadores: a base original e reconstruida repetindo cada linha (1 + copias_retiradas) vezes
cop = ler(P_PUB, ['data_compra', 'cod_catmat', 'unidade_chave', 'preco_unitario', 'quantidade', 'valor_total',
                  'cnpj_fornecedor', 'copias_retiradas'])
cop = (cop.filter(pl.col('data_compra').is_not_null() & (pl.col('preco_unitario') > 0) & pl.col('cod_catmat').is_not_null())
          .with_columns(pl.col('data_compra').dt.year().alias('ano'), pl.col('data_compra').dt.month().alias('mes'),
                        (pl.col('cod_catmat') + '|' + pl.col('unidade_chave').fill_null('')).alias('item'))
          .join(ipca, on=['ano', 'mes'], how='left')   # D16/D20: régua sobre o preço real; valor em reais de dez/2023
          .with_columns((pl.col('preco_unitario') * pl.col('fator_deflacao')).alias('preco_real'),
                        (pl.col('valor_total') * pl.col('fator_deflacao')).alias('valor_real')))
antes = (cop.with_columns(pl.int_ranges(0, pl.col('copias_retiradas') + 1).alias('_k')).explode('_k').drop('_k'))
def resumo(df, rotulo):
    w = (df.filter((pl.col('valor_total') > 0) & (pl.col('quantidade') > 0))
           .with_columns(pl.col('preco_real').median().over(['item', 'ano']).alias('p_med'),
                         pl.col('quantidade').median().over(['item', 'ano']).alias('q_med'),
                         pl.len().over(['item', 'ano']).alias('n_item')))
    w = w.filter(cnpjf.is_not_null() & (cnpjf.str.len_chars() > 0) & (cnpjf != '00000000000000'))
    c = hhi(w, 'compras').filter(pl.col('n') >= MIN_COMPRAS); vv = hhi(w.filter(~flag(REGUA_PRECO)), 'valor').filter(pl.col('n') >= MIN_COMPRAS)
    tot = {'base': rotulo, 'ano': 'TOTAL', 'registros': df.height, 'classificadas_compras': c.height,
           'alta_compras_pct': round((c['hhi'] > LIM_ALTO).mean() * 100, 1), 'classificadas_valor': vv.height,
           'alta_valor_pct': round((vv['hhi'] > LIM_ALTO).mean() * 100, 1)}
    porano = [{'base': rotulo, 'ano': str(a), 'registros': df.filter(pl.col('ano') == a).height,
               'classificadas_compras': c.filter(pl.col('ano') == a).height,
               'alta_compras_pct': round((c.filter(pl.col('ano') == a)['hhi'] > LIM_ALTO).mean() * 100, 1),
               'classificadas_valor': vv.filter(pl.col('ano') == a).height,
               'alta_valor_pct': round((vv.filter(pl.col('ano') == a)['hhi'] > LIM_ALTO).mean() * 100, 1)} for a in [2021, 2022, 2023]]
    return porano + [tot]
t12e = pl.DataFrame(resumo(antes, 'com duplicatas (antes)') + resumo(cop, 'sem duplicatas (atual)'))
print(f'\n12e. EFEITO DA RETIRADA (base publica, preco > 0; >= {MIN_COMPRAS} compras; regua {REGUA_PRECO}x no HHI por valor)')
print(t12e)
print('Leitura: a retirada quase nao altera 2021-2022; em 2023 as copias inflavam a contagem de compras de poucos')
print('compradores e, com isso, a concentracao por numero de compras do ano.')
salvar(t12e, '12e_efeito_da_retirada')

print('\n12f. RESSALVA — linhas iguais em tudo, exceto a data de insercao, NAO foram retiradas: '
      f"{t12a['iguais_exceto_data_insercao'].sum():,} linhas "
      f"({t12a['iguais_exceto_data_insercao'].sum() / t12a['linhas_fonte'].sum():.2%} da fonte). Podem ser a mesma compra"
      ' inserida duas vezes ou compras distintas com os mesmos dados; a base nao permite distinguir.')


12a. DUPLICATAS EXATAS POR ANO (todas as colunas do arquivo de origem)
shape: (15, 7)
┌──────┬──────────────┬───────────────────┬────────────────┬─────────────────────────────┬─────────────────────┬─────────┐
│ ano  ┆ linhas_fonte ┆ duplicatas_exatas ┆ pct_duplicatas ┆ iguais_exceto_data_insercao ┆ retiradas_rotina_03 ┆ confere │
│ ---  ┆ ---          ┆ ---               ┆ ---            ┆ ---                         ┆ ---                 ┆ ---     │
│ i64  ┆ i64          ┆ i64               ┆ f64            ┆ i64                         ┆ i64                 ┆ bool    │
╞══════╪══════════════╪═══════════════════╪════════════════╪═════════════════════════════╪═════════════════════╪═════════╡
│ 2009 ┆ 13494        ┆ 31                ┆ 0.23           ┆ 104                         ┆ 31                  ┆ true    │
│ 2010 ┆ 19069        ┆ 3                 ┆ 0.02           ┆ 21                          ┆ 3                   ┆ true    │
│ 2011 ┆ 13611        ┆ 0                 ┆ 0.0      

## 13. Indicadores do painel — os testes que os sustentam
**Decisão.** O painel (rotina 12) traz quatro indicadores descritivos calculados sobre a MESMA base do texto:
diferença de preço entre compradores, preço por modalidade, época da compra e repetição do fornecedor (os fornecedores
"estreantes" foram avaliados e descartados, D52).
**Testes.** (a) a diferença entre compradores não é efeito do lote nem dos valores implausíveis; (b) a diferença por
modalidade, antes e depois de comparar com o pregão no mesmo item, ano e faixa de lote; (c) a participação do 1º semestre
em todos os anos, de 2009 a 2023, cada um com o seu número de compras (D51): na maior parte dos anos passa da metade, com
exceções (2009, 2010 e 2014), e 2023 fica inflado pelo volume baixo de abril a dezembro (D21); (d) duas formas de medir a
repetição do fornecedor; (e) por que a comparação com
o preço regulado da CMED (arquivos de 2017 e 2021) foi descartada.
(f) o mínimo de 10 compras do item-ano para a diferença entre compradores (D44): abaixo de 10, P90/P10 se confunde com maior ÷ menor preço (arquivo 13f). O mínimo de 7 compras da concentração tem a prova no arquivo 06a (seção 6).

In [18]:
# régua: a marca da rotina 04B (D16, sobre o preço real), a mesma do painel
v13 = (pub.filter((pl.col('quantidade') > 0) & (pl.col('valor_total') > 0))
          .with_columns(pl.col('valor_implausivel').fill_null(False).alias('implaus')))
ok13 = v13.filter(~pl.col('implaus')).with_columns(pl.len().over(['item', 'ano']).alias('n_ia'),
        (pl.col('preco_real').log() - pl.col('preco_real').log().median().over(['item', 'ano'])).alias('dp'),
        (pl.col('quantidade').log() - pl.col('quantidade').log().median().over(['item', 'ano'])).alias('dq'))
ok13 = ok13.with_columns(pl.col('faixa_lote').alias('ql'))   # a faixa de lote da rotina 04B (D31, D53), a mesma do painel

MIN_DISPERSAO  = 10   # D44: mínimo de compras no item-ano para a diferença entre compradores (P90/P10; teste 13f)
MIN_MODALIDADE = 3    # D45: mínimo de compras no item-ano para comparar modalidades (IN SEGES/ME 65/2021, art. 6º)
print(f'13a. DIFERENÇA DE PREÇO ENTRE COMPRADORES (item-ano com >= {MIN_DISPERSAO} compras)')
def p90p10(df, col='preco_real'):
    return df.group_by('item', 'ano').agg((pl.col(col).quantile(.9, 'linear') / pl.col(col).quantile(.1, 'linear')).alias('r'))['r'].median()
com_impl = v13.with_columns(pl.len().over(['item', 'ano']).alias('n_ia')).filter(pl.col('n_ia') >= MIN_DISPERSAO)
e13 = ok13.filter(pl.col('n_ia') >= MIN_DISPERSAO).with_columns((pl.col('dp') - pl.col('dp').median().over('ql')).alias('res'))
dd = e13.group_by('item', 'ano').agg((pl.col('preco_real').quantile(.9, 'linear') / pl.col('preco_real').quantile(.1, 'linear')).alias('r'),
                                      (pl.col('preco_real').quantile(.75, 'linear') / pl.col('preco_real').quantile(.25, 'linear')).alias('r75'),
                                      (pl.col('res').quantile(.9, 'linear') - pl.col('res').quantile(.1, 'linear')).exp().alias('r_lote'))
t13a = pl.DataFrame([
    {'medida': 'P90/P10, com valores implausíveis', 'mediana': round(p90p10(com_impl), 2)},
    {'medida': 'P90/P10, sem valores implausíveis (publicada)', 'mediana': round(dd['r'].median(), 2)},
    {'medida': 'P75/P25, sem valores implausíveis', 'mediana': round(dd['r75'].median(), 2)},
    {'medida': 'P90/P10, retirado o efeito da faixa de lote', 'mediana': round(dd['r_lote'].median(), 2)},
    {'medida': '% de itens-ano com P90 > 2 x P10', 'mediana': round((dd['r'] > 2).mean() * 100, 1)}])
print(t13a); salvar(t13a, '13a_dispersao_robustez')

print(f'\n13b. PREÇO POR MODALIDADE (item-ano com >= {MIN_MODALIDADE} compras, D45; sem valores implausíveis)')
w13 = ok13.filter(pl.col('n_ia') >= MIN_MODALIDADE)
pct = lambda e: ((e.exp() - 1) * 100).round(1)
br13 = w13.group_by('modalidade_grupo').agg(pl.len().alias('compras'), pct(pl.col('dp').median()).alias('bruto_pct'))
pg = w13.filter(pl.col('modalidade_grupo') == 'Pregão')
def pareado(chave, nome):
    a = pg.group_by(chave).agg(pl.col('dp').median().alias('dp_pg'))
    return (w13.filter(pl.col('modalidade_grupo') != 'Pregão').group_by(['modalidade_grupo', *chave]).agg(pl.col('dp').median().alias('dp_m'))
              .join(a, on=chave).group_by('modalidade_grupo').agg(pct((pl.col('dp_m') - pl.col('dp_pg')).median()).alias(nome)))
t13b = (br13.join(pareado(['item', 'ano'], 'vs_pregao_mesmo_item_ano_pct'), on='modalidade_grupo', how='left')
            .join(pareado(['item', 'ano', 'ql'], 'vs_pregao_mesmo_item_ano_lote_pct'), on='modalidade_grupo', how='left')
            .sort('compras', descending=True))
print(t13b); salvar(t13b, '13b_modalidade_controles')
print('Leitura: parte da diferença bruta vem do lote menor das compras fora do pregão; comparando o mesmo item, ano e faixa')
print('de lote, a diferença diminui, mas não desaparece (publicada: última coluna).')

print('\n13c. ÉPOCA DA COMPRA — % das compras no 1º semestre, por ano (data da compra informada pela instituição)')
# D51: todos os anos, cada um com o seu número de compras (antes: 2015–2022, recorte sem decisão)
t13c = (pub.group_by('ano')
           .agg(pl.len().alias('compras'), ((pl.col('mes') <= 6).mean() * 100).round(1).alias('pct_1o_semestre')).sort('ano'))
print(t13c); salvar(t13c, '13c_sazonalidade_por_ano')
print(f"Faixa: {t13c['pct_1o_semestre'].min()}% a {t13c['pct_1o_semestre'].max()}% | anos abaixo de 50%: "
      f"{t13c.filter(pl.col('pct_1o_semestre') < 50)['ano'].to_list()} | 2023 com a ressalva da D21")

print('\n13d. REPETIÇÃO DO FORNECEDOR (mesmo comprador e item em anos seguidos)')
ci = pub.group_by('cnpj_instituicao', 'item', 'ano').agg(pl.col('cnpj_fornecedor').unique().alias('fs'),
                                                          pl.col('cnpj_fornecedor').mode().sort().first().alias('principal'))
jj = ci.join(ci.with_columns((pl.col('ano') + 1).alias('ano')), on=['cnpj_instituicao', 'item', 'ano'], suffix='_ant')
t13d = pl.DataFrame([{'medida': 'algum fornecedor do ano anterior vence de novo (publicada)',
                      'pct': round(jj.select(pl.col('fs').list.set_intersection(pl.col('fs_ant')).list.len().gt(0).mean()).item() * 100, 1)},
                     {'medida': 'o fornecedor principal é o mesmo do ano anterior',
                      'pct': round((jj['principal'] == jj['principal_ant']).mean() * 100, 1)}]).with_columns(pl.lit(jj.height).alias('pares'))
print(t13d); salvar(t13d, '13d_repeticao_fornecedor')

print('\n13e. PREÇO REGULADO DA CMED — por que NÃO entra no painel')
import re
def num(c):
    b = pl.col(c).str.replace_all(' ', '').str.strip_chars()
    return pl.when(b.str.contains(',', literal=True)).then(b.str.replace_all('.', '', literal=True).str.replace(',', '.', literal=True)).otherwise(b).cast(pl.Float64, strict=False)
linhas = []
for a in (2017, 2021):
    r = pl.read_csv(P_CSV / f'BPS_{a}.csv', infer_schema_length=0, encoding='utf8-lossy').unique()
    r = r.rename({c: re.sub(r'\s+', ' ', c.replace('\xa0', ' ')).strip() for c in r.columns})
    pc = next(c for c in r.columns if c in ('Preço Unitário', 'Unitário', 'Pago'))
    x = r.with_columns(num('CMED - Preço Regulado').alias('cmed'), num(pc).alias('p')).filter(pl.col('cmed') > 0)
    x = x.with_columns((pl.col('p') / pl.col('cmed')).alias('razao'))
    comp = sorted(r['Competência CMED'].drop_nulls().unique().to_list())
    linhas.append({'arquivo': a, 'linhas_com_cmed': x.height, 'competencia_cmed': ' | '.join(comp),
                   'razao_mediana_pago_cmed': round(x['razao'].median(), 3),
                   'pct_acima_do_teto': round((x['razao'] > 1.0001).mean() * 100, 1),
                   'pct_mais_de_3x_o_teto': round((x['razao'] > 3).mean() * 100, 1),
                   'pct_abaixo_de_5pct_do_teto': round((x['razao'] < 0.05).mean() * 100, 1)})
t13e = pl.DataFrame(linhas)
print(t13e); salvar(t13e, '13e_cmed_descartada')
print('Leitura: o preço regulado informado é de uma competência POSTERIOR às compras (a mesma para o ano inteiro), só existe')
print('em 2 dos 15 arquivos e tem casos de ordem de grandeza incompatível (sinal de unidade diferente). Não sustenta uma série.')


13a. DIFERENÇA DE PREÇO ENTRE COMPRADORES (item-ano com >= 10 compras)
shape: (5, 2)
┌───────────────────────────────────────────────┬─────────┐
│ medida                                        ┆ mediana │
│ ---                                           ┆ ---     │
│ str                                           ┆ f64     │
╞═══════════════════════════════════════════════╪═════════╡
│ P90/P10, com valores implausíveis             ┆ 2.64    │
│ P90/P10, sem valores implausíveis (publicada) ┆ 2.4     │
│ P75/P25, sem valores implausíveis             ┆ 1.51    │
│ P90/P10, retirado o efeito da faixa de lote   ┆ 2.34    │
│ % de itens-ano com P90 > 2 x P10              ┆ 68.4    │
└───────────────────────────────────────────────┴─────────┘
  -> Auditoria/13a_dispersao_robustez.csv

13b. PREÇO POR MODALIDADE (item-ano com >= 3 compras, D45; sem valores implausíveis)
shape: (8, 5)
┌────────────────────┬─────────┬───────────┬──────────────────────────────┬───────────────────────────────────┐
│

In [19]:
# 13f. Mínimo de 10 compras para a diferença entre compradores (D44). P10 e P90 deixam de fora os 10% de cada
# extremo; com menos de 10 compras, 10% é menos de uma compra e a razão P90/P10 se confunde com maior ÷ menor preço.
LIMITES_FAIXAS_N = [2, 4, 6, 9, 19]   # só a apresentação do teste em faixas de número de compras; o mínimo adotado é 10 (D44)
q = lambda x: pl.col('preco_real').quantile(x, 'linear')
g13 = ok13.group_by('item', 'ano').agg(pl.len().alias('compras'), (q(.9) / q(.1)).alias('p90_p10'),
                                       (pl.col('preco_real').max() / pl.col('preco_real').min()).alias('maior_menor'))
t13f = (g13.with_columns(pl.col('compras').cut(LIMITES_FAIXAS_N, labels=['1 a 2', '3 a 4', '5 a 6', '7 a 9', '10 a 19', '20 ou mais'])
                          .alias('faixa_compras'))
           .group_by('faixa_compras').agg(pl.col('compras').min().alias('_ordem'), pl.len().alias('itens_ano'), pl.col('compras').sum().alias('compras'),
                                          pl.col('p90_p10').median().round(2).alias('p90_p10_mediana'),
                                          pl.col('maior_menor').median().round(2).alias('maior_menor_mediana'),
                                          ((pl.col('p90_p10') / pl.col('maior_menor')).median() * 100).round(0).alias('p90_p10_em_pct_do_maior_menor'))
           .sort('_ordem').drop('_ordem'))
print('13f. P90/P10 POR NÚMERO DE COMPRAS NO ITEM-ANO (sem valores implausíveis)'); print(t13f)
print(f"Com 10 compras ou mais: {g13.filter(pl.col('compras') >= 10)['compras'].sum() / g13['compras'].sum():.1%} das compras")
salvar(t13f, '13f_minimo_p90_p10')


13f. P90/P10 POR NÚMERO DE COMPRAS NO ITEM-ANO (sem valores implausíveis)
shape: (6, 6)
┌───────────────┬───────────┬─────────┬─────────────────┬─────────────────────┬───────────────────────────────┐
│ faixa_compras ┆ itens_ano ┆ compras ┆ p90_p10_mediana ┆ maior_menor_mediana ┆ p90_p10_em_pct_do_maior_menor │
│ ---           ┆ ---       ┆ ---     ┆ ---             ┆ ---                 ┆ ---                           │
│ cat           ┆ u32       ┆ u32     ┆ f64             ┆ f64                 ┆ f64                           │
╞═══════════════╪═══════════╪═════════╪═════════════════╪═════════════════════╪═══════════════════════════════╡
│ 1 a 2         ┆ 41675     ┆ 54976   ┆ 1.0             ┆ 1.0                 ┆ 100.0                         │
│ 3 a 4         ┆ 11847     ┆ 40318   ┆ 1.72            ┆ 1.98                ┆ 88.0                          │
│ 5 a 6         ┆ 5980      ┆ 32434   ┆ 2.07            ┆ 2.6                 ┆ 82.0                          │
│ 7 a 9         

## 14. Quem domina o item: a medida do mapa de oportunidades (D28 e D29)
**Decisões.** D28: a fatia do maior fornecedor é medida pelo **valor** (medida principal) e pelo **número de compras** (segunda medida); cada ponto do mapa é um **item em um ano** com 7 compras ou mais naquele ano, o mesmo recorte do HHI (D7). D29: item grande = item-ano com compras iguais ou acima do corte do quintil superior do ano (percentil 80; mesma convenção de quintis da análise de escala); fornecedor com posição dominante = fatia de 20% ou mais (Lei 12.529/2011, art. 36, §2º, como analogia declarada); cor do mapa por UF centrada em 1 (Silva et al., 2025).

**Lastro das medidas.** Resolução CMED nº 1/2015, Anexo, item 4.2.1 ("faturamento em R$"); CADE (2016), §2.4.1 (unidades ou valor; diversas formas de mensuração); Pliatsidis (2024) (número de contratos).

**Teste.** Todos os anos, de 2009 a 2023, cada um separado, como o mapa da rotina 12 (até 19/10/2026 o teste olhava só 2021–2023, recorte herdado do esboço antigo, sem decisão). (a) itens no mapa por ano e corte do quintil superior; (b) itens com fornecedor de posição dominante (20% ou mais) por medida; (c) concordância entre as medidas; (d) conferência de que a régua de implausíveis alcança todas as compras do mapa: o mapa seleciona o item-ano por TODAS as compras (`compras_item_ano`, como na rotina 12), e a régua só confere item-ano com 3 compras avaliáveis ou mais (com fornecedor, quantidade e valor).

In [20]:
# régua de valores implausíveis: a marca da rotina 04B (D16), a mesma das rotinas 06 e 12
cnpj14 = pl.col('cnpj_fornecedor').cast(pl.Utf8).str.strip_chars()
b14 = (pub.filter(cnpj14.is_not_null() & (cnpj14.str.len_chars() > 0) & (cnpj14 != '00000000000000')
                  & (pl.col('quantidade') > 0) & (pl.col('valor_total') > 0))
          .with_columns(pl.len().over(['item', 'ano']).alias('n_ia'),                       # compras avaliáveis pela régua
                        pl.col('valor_implausivel').fill_null(False).alias('implaus'))   # régua da rotina 04B (D16)
          # recorte do mapa igual ao da rotina 12: item-ano com 7 compras ou mais, contando TODAS as compras (D7, D28)
          .filter(pl.col('compras_item_ano') >= MIN_COMPRAS))   # todos os anos, como o mapa da rotina 12 (D28)
ok14 = b14.filter(~pl.col('implaus'))

def fatia_lider(df, peso, nome):
    w = df.group_by('item', 'ano', 'cnpj_fornecedor').agg(peso.alias('w'))
    return w.group_by('item', 'ano').agg((pl.col('w').max() / pl.col('w').sum()).alias(nome))

f14 = (b14.group_by('item', 'ano').agg(pl.len().alias('compras_ia'), (~pl.col('implaus')).sum().alias('validas_ia'))
          .join(fatia_lider(ok14, pl.col('valor_real').sum(), 'valor'), on=['item', 'ano'], how='left')
          .join(fatia_lider(b14, pl.len().cast(pl.Float64), 'compras'), on=['item', 'ano'], how='left')
          # D54: a fatia por valor só existe com 7 compras válidas ou mais (a mesma regra da rotina 12 e do HHI por valor)
          .with_columns(pl.when(pl.col('validas_ia') >= MIN_COMPRAS).then(pl.col('valor')).otherwise(None).alias('valor')))
LIM_DOMINANTE = 0.20     # Lei 12.529/2011, art. 36, §2º (analogia declarada, D29)

t14a = (f14.group_by('ano').agg(pl.len().alias('itens_no_mapa'),
                                pl.col('compras_ia').quantile(CORTE_ITEM_GRANDE, 'linear').alias('corte_quintil_superior_compras'),
                                (pl.col('compras_ia') >= pl.col('compras_ia').quantile(CORTE_ITEM_GRANDE, 'linear')).sum().alias('itens_grandes'),
                                (pl.col('valor') >= LIM_DOMINANTE).sum().alias('dominante_pelo_valor'),
                                (pl.col('compras') >= LIM_DOMINANTE).sum().alias('dominante_pelas_compras'))
          .sort('ano'))
print('14a. Itens do mapa por ano (item-ano com 7 compras ou mais) e fornecedor com posição dominante (20% ou mais)')
print(t14a); salvar(t14a, '14a_dominio_por_ano')

d14 = f14.drop_nulls(['valor', 'compras'])
t14b = (d14.group_by('ano').agg(pl.len().alias('itens'),
                                (((pl.col('valor') >= LIM_DOMINANTE) == (pl.col('compras') >= LIM_DOMINANTE)).mean() * 100).round(1)
                                  .alias('pct_mesma_classificacao_20pct'),
                                pl.corr('valor', 'compras', method='spearman').round(3).alias('spearman_fatia_lider'),
                                (pl.col('valor').median() * 100).round(1).alias('fatia_mediana_lider_valor_pct'),
                                (pl.col('compras').median() * 100).round(1).alias('fatia_mediana_lider_compras_pct'))
           .sort('ano'))
print('\n14b. Concordância entre valor e número de compras'); print(t14b); salvar(t14b, '14b_dominio_concordancia')

# antes o teste procurava n_ia < 3 depois de já exigir n_ia >= 7 e nunca podia falhar. Agora o mapa é
# selecionado por todas as compras e o teste olha as compras AVALIÁVEIS de cada item-ano do mapa
fora = b14.filter(pl.col('n_ia') < MIN_REG_REGUA).height
# conferir também as compras VÁLIDAS para valor de cada item-ano do mapa (D54)
abaixo7 = f14.filter(pl.col('validas_ia') < MIN_COMPRAS)
t14c = pl.DataFrame({'compras_no_mapa': [b14.height], 'compras_fora_do_alcance_da_regua': [fora],
                     'compras_marcadas_implausiveis': [int(b14['implaus'].sum())],
                     'item_ano_com_menos_de_7_validas': [abaixo7.height],
                     'desses_com_fatia_por_valor': [int(abaixo7['valor'].is_not_null().sum())]})
print('\n14c. Alcance da régua no mapa'); print(t14c); salvar(t14c, '14c_dominio_alcance_regua')
assert fora == 0, 'Há compras do mapa fora do alcance da régua: o recorte não é o da D7'
assert abaixo7['valor'].is_null().all(), 'Há fatia por valor com menos de 7 compras válidas: contraria a D54'


14a. Itens do mapa por ano (item-ano com 7 compras ou mais) e fornecedor com posição dominante (20% ou mais)
shape: (15, 6)
┌──────┬───────────────┬────────────────────────────────┬───────────────┬──────────────────────┬─────────────────────────┐
│ ano  ┆ itens_no_mapa ┆ corte_quintil_superior_compras ┆ itens_grandes ┆ dominante_pelo_valor ┆ dominante_pelas_compras │
│ ---  ┆ ---           ┆ ---                            ┆ ---           ┆ ---                  ┆ ---                     │
│ i32  ┆ u32           ┆ f64                            ┆ u32           ┆ u32                  ┆ u32                     │
╞══════╪═══════════════╪════════════════════════════════╪═══════════════╪══════════════════════╪═════════════════════════╡
│ 2009 ┆ 192           ┆ 10.0                           ┆ 56            ┆ 180                  ┆ 160                     │
│ 2010 ┆ 480           ┆ 18.0                           ┆ 99            ┆ 477                  ┆ 374                     │
│ 2011 ┆ 382   

## 15. Classe CATMAT e exclusão dos códigos fora da saúde (D40, D41)
**Decisões.** D40: o tipo de produto é a classe CATMAT oficial. D41: a classe vem do BPS 2018–2020 e, para os demais
códigos, do catálogo oficial (consulta datada, rotina 03D); saem os códigos fora das 16 classes de saúde do Ministério da Saúde.
**Testes (gravados pelas rotinas 03D e 04B).** Concordância entre a classe do BPS e a do catálogo; cobertura por ano;
códigos retirados; e, aqui, a conferência de que a base analítica só tem classes de saúde.

In [21]:
CLASSES_SAUDE_MS = ['6505', '6508', '6509', '6510', '6515', '6520', '6525', '6530', '6532',
                    '6540', '6545', '6550', '6640', '6810', '6840', '8940']   # D41
for nome in sorted(f.stem for f in SAIDA.glob('03[d-g]_*.csv')) + ['BA1_classe_e_exclusao_por_ano']:
    arq = SAIDA / f'{nome}.csv'
    if arq.exists():
        print(f'\n{nome} (gravado pela rotina {"04B" if nome.startswith("BA") else "03D"}):'); print(pl.read_csv(arq, infer_schema_length=0).head(20))
assert (SAIDA / 'BA1_classe_e_exclusao_por_ano.csv').exists(), 'rode a rotina 04B antes'
fora = pub.filter(~pl.col('classe_catmat').is_in(CLASSES_SAUDE_MS)).height
print(f'\nConferência: compras da base analítica fora das classes de saúde = {fora} (deve ser 0)')
assert fora == 0
t15 = (pub.group_by('classe_catmat', 'nome_classe').agg(pl.len().alias('compras'), pl.col('item').n_unique().alias('itens'))
          .sort('compras', descending=True))
print(t15); salvar(t15, '15_compras_por_classe')


03d_classe_bps_x_catalogo (gravado pela rotina 03D):
shape: (4, 3)
┌───────────────────────────┬─────────┬─────────┐
│ medida                    ┆ codigos ┆ compras │
│ ---                       ┆ ---     ┆ ---     │
│ str                       ┆ str     ┆ str     │
╞═══════════════════════════╪═════════╪═════════╡
│ codigos com classe no BPS ┆ 10837   ┆ 808021  │
│ encontrados no catalogo   ┆ 10837   ┆ 808021  │
│ mesma classe              ┆ 10821   ┆ 807587  │
│ classe diferente          ┆ 16      ┆ 434     │
└───────────────────────────┴─────────┴─────────┘

03e_divergencias_classe (gravado pela rotina 03D):
shape: (16, 7)
┌────────────┬────────────┬─────────────────┬───────────────────────────────────────────────────────────────┬────────────┬─────────┬───────────────────────────────────────────────────────────────┐
│ cod_catmat ┆ classe_bps ┆ classe_catalogo ┆ nome_classe                                                   ┆ item_ativo ┆ compras ┆ descricao_bps                      

## 16. Curvas de robustez da fase 2 (D30, D31, D34, D47)
**Decisões.** Em vez de cenários escolhidos, as curvas completas: D30, índice de 2023 por exigência de permanência do item
(2 a 15 anos); D34, desenhos A e C da correlação por mínimo de anos; D47, teste de composição do comprador por ano de início.
As curvas são calculadas pelas rotinas 05 e 09; aqui são copiadas para a pasta Auditoria, ao lado dos demais testes.
**Atualização 25/10/2026 (D45, complemento).** O 16a (cópia do `robustez_indice` da rotina 05) passa a trazer também a curva com 3 compras válidas (`indice_2023_min3`). O 16b2 é o teste de composição (D47) com a regra de 3, contada dentro de cada recorte; o 16b fica como referência da versão sem mínimo.


In [22]:
copias = {'16a_curva_permanencia_d30': BASE / 'Base Preco Real - Publico' / 'robustez_indice.csv',
          '16b_composicao_ano_inicio_d47': BASE / 'Base Preco Real - Publico' / 'teste_composicao_curva_inicio.csv',
          '16b2_composicao_ano_inicio_d47_min3': BASE / 'Base Preco Real - Publico' / 'teste_composicao_curva_inicio_min3.csv',   # D45 (compl. 25/10/2026)
          '16c_curva_desenho_a_d34': BASE / 'Base Correlacao - Publico' / 'correlacao_curva_a.csv',
          '16d_curva_desenho_c_d34': BASE / 'Base Correlacao - Publico' / 'correlacao_curva_c.csv',
          '16e_trajetoria_descartada_d48': BASE / 'Base Correlacao - Publico' / 'descartada_trajetoria_curva.csv'}
for nome, arq in copias.items():
    assert arq.exists(), f'{arq} não encontrado: rode as rotinas 05 e 09 antes'
    t = pl.read_csv(arq)
    print(f'\n{nome}  (de {arq.parent.name}/{arq.name})'); print(t)
    salvar(t, nome)


16a_curva_permanencia_d30  (de Base Preco Real - Publico/robustez_indice.csv)
shape: (15, 6)
┌─────────────┬──────────────────────────────┬─────────────┬────────────────────┬──────────────────┬─────────────────────────┐
│ minimo_anos ┆ cenario                      ┆ indice_2023 ┆ itens_no_maior_elo ┆ indice_2023_min3 ┆ itens_no_maior_elo_min3 │
│ ---         ┆ ---                          ┆ ---         ┆ ---                ┆ ---              ┆ ---                     │
│ i64         ┆ str                          ┆ f64         ┆ i64                ┆ f64              ┆ i64                     │
╞═════════════╪══════════════════════════════╪═════════════╪════════════════════╪══════════════════╪═════════════════════════╡
│ 1           ┆ todos os itens               ┆ 77.1        ┆ 7200               ┆ 86.9             ┆ 3948                    │
│ 2           ┆ presentes em 2 anos ou mais  ┆ 77.1        ┆ 7200               ┆ 86.9             ┆ 3948                    │
│ 3           ┆ p

### 16f. Número de grupos (D31)
**Decisão.** D31: quintis (5 grupos de 20%) são a convenção de grupos do trabalho (escala, faixa de lote, item grande, desenho B).
**Teste.** A escala e o desenho B refeitos com 3, 4, 5 e 10 grupos, do menor para o maior grupo. O teste que apoiou a D31
(13/10/2026) foi feito na base anterior e não ficou gravado; esta célula o refaz na base analítica e grava em
Auditoria/16f. Com 5 grupos, os números têm de ser iguais aos publicados pelas rotinas 08 e 09.

In [23]:
# 16f. D31: por que quintis. Refaz a escala (rotina 04B/08) e o desenho B (rotina 09) com 3, 4, 5 e 10 grupos.
# O teste original (13/10/2026) foi feito na base anterior e não ficou gravado; esta célula o refaz na base analítica.
GRUPOS_TESTE = [3, 4, 5, 10]   # D31: os números de grupos comparados no teste de 13/10/2026 (5 = quintis, a convenção)
ARQ_HHI16 = BASE / 'Base HHI - Publico' / 'hhi_concentracao.parquet'   # rotina 06 (D17)
assert ARQ_HHI16.exists(), 'Rode a rotina 06 antes'

# escala: mesmo cálculo da rotina 04B (desvio do lote em relação ao item no ano; grupos sobre todas as compras), D53
esc = (pub_val.filter((pl.col('quantidade') > 0) & (pl.col('compras_validas_item_ano') >= MIN_MEDIANA))
              .with_columns((pl.col('quantidade').log() - pl.col('quantidade').log().median().over(['item', 'ano'])).alias('dq'),
                            pl.col('preco_rel_mediana').log().alias('dp')))
# desenho B: mesmo painel da rotina 09 (item-ano com 7 compras ou mais, D7; preço = mediana do item no ano)
preco16 = (pl.read_parquet(ARQ_BASE, columns=['ano', 'item', 'mediana_item_ano', 'valor_implausivel'])
           .filter(~pl.col('valor_implausivel').fill_null(False))   # D16, como na rotina 09
           .group_by(['ano', 'item']).agg(pl.col('mediana_item_ano').first().alias('preco')))
hhi16 = pl.read_parquet(ARQ_HHI16)
pares = {}
for m, (col_h, col_n) in {'compras': ('hhi_compras', 'n_compras'), 'valor': ('hhi_valor', 'n_compras_valor')}.items():
    pm = (hhi16.filter(pl.col(col_n) >= MIN_COMPRAS).select(['ano', 'item', pl.col(col_h).alias('hhi')])
               .join(preco16, on=['ano', 'item'], how='inner').filter(pl.col('preco') > 0).sort(['item', 'ano']))
    pares[m] = (pm.with_columns((pl.col('hhi') - pl.col('hhi').shift(1).over('item')).alias('dq'),
                                (pl.col('preco') / pl.col('preco').shift(1).over('item') - 1).alias('dp'),
                                (pl.col('ano') - pl.col('ano').shift(1).over('item')).alias('gap'))
                  .filter((pl.col('gap') == 1) & pl.col('dq').is_not_null() & pl.col('dp').is_not_null()))

def por_grupos(df, k, transf):
    rot = [f'G{i:02d}' for i in range(1, k + 1)]
    return (df.with_columns(pl.col('dq').qcut(k, labels=rot, allow_duplicates=True).cast(pl.Utf8).alias('grupo'))
              .group_by('grupo').agg(transf(pl.col('dp').median()).alias('resultado_pct'), pl.len().alias('n')).sort('grupo'))

linhas16 = []
for estudo, df, transf in [('escala (desvio de preço por faixa de lote)', esc, lambda e: ((e.exp() - 1) * 100).round(1)),
                           ('desenho B, HHI por compras (variação do preço por faixa de variação do HHI)', pares['compras'], lambda e: (e * 100).round(2)),
                           ('desenho B, HHI por valor (variação do preço por faixa de variação do HHI)', pares['valor'], lambda e: (e * 100).round(2))]:
    for k in GRUPOS_TESTE:
        t = por_grupos(df, k, transf)
        r = t['resultado_pct'].to_list()
        linhas16.append({'estudo': estudo, 'grupos': k, 'primeiro_grupo_pct': r[0], 'ultimo_grupo_pct': r[-1],
                         'sempre_na_mesma_direcao': all(a >= b for a, b in zip(r, r[1:])),
                         'grupos_fora_da_ordem': sum(a < b for a, b in zip(r, r[1:])),
                         'todos_os_grupos': ' | '.join(f'{x:+.2f}' for x in r), 'compras_ou_pares': int(t['n'].sum())})
t16f = pl.DataFrame(linhas16)
pl.Config.set_fmt_str_lengths(120); pl.Config.set_tbl_rows(-1)
print('16f. D31 — o mesmo resultado com 3, 4, 5 e 10 grupos? (do menor para o maior grupo)'); print(t16f)
salvar(t16f, '16f_numero_de_grupos_d31')

# conferência: com 5 grupos, os números têm de ser os publicados pelas rotinas 08 e 09
q08 = pl.read_csv(BASE / 'Base Escala - Publico' / 'escala_quintis.csv')['desvio_preco_pct'].to_list()
b09 = pl.read_csv(BASE / 'Base Correlacao - Publico' / 'correlacao_desenho_b.csv')
assert por_grupos(esc, 5, lambda e: ((e.exp() - 1) * 100).round(1))['resultado_pct'].to_list() == q08, 'escala com 5 grupos difere da rotina 08'
for m in ['compras', 'valor']:
    assert (por_grupos(pares[m], 5, lambda e: (e * 100).round(2))['resultado_pct'].to_list()
            == b09.filter(pl.col('medida') == m)['var_preco_mediana_pct'].to_list()), f'desenho B ({m}) com 5 grupos difere da rotina 09'
print('Conferência: com 5 grupos, escala igual à rotina 08 e desenho B igual à rotina 09.')


16f. D31 — o mesmo resultado com 3, 4, 5 e 10 grupos? (do menor para o maior grupo)
shape: (12, 8)
┌──────────────────────────────────────────────────┬────────┬────────────────────┬──────────────────┬─────────────────────────┬──────────────────────┬──────────────────────────────────────────────────┬──────────────────┐
│ estudo                                           ┆ grupos ┆ primeiro_grupo_pct ┆ ultimo_grupo_pct ┆ sempre_na_mesma_direcao ┆ grupos_fora_da_ordem ┆ todos_os_grupos                                  ┆ compras_ou_pares │
│ ---                                              ┆ ---    ┆ ---                ┆ ---              ┆ ---                     ┆ ---                  ┆ ---                                              ┆ ---              │
│ str                                              ┆ i64    ┆ f64                ┆ f64              ┆ bool                    ┆ i64                  ┆ str                                              ┆ i64              │
╞════════════════

### 16g. Preço na página do item: faixa, mapa, barras por estado e preço por fornecedor (P2.2, P2.5, P2.6, P2.9; complementos da D46 e da D58)
**Decisão.** A faixa da P2.2 mostra a mediana do preço real das compras válidas que ficam no filtro (período, um ou mais
estados, comprador), com o número de compras ao lado. A compra válida vem da rotina 12 (`compra_valida = "Sim"`, D61); no
Power BI fica só a contagem: com menos de 3 compras no filtro, a faixa fica vazia (D45; exceção declarada, como o 7 da D58).
**Teste.** Com um ano, a mesma conta tem de reproduzir as medianas prontas da rotina 12: país (`ind_dispersao_item_ano` e a
coluna `mediana_item_ano` da `fato_compras`) e estado (`ind_item_estado_ano`, inclusive o número de compras e a razão estado ÷ país das P2.5/P2.6, que no painel é
mediana do estado no período ÷ mediana do país no período). As tabelas prontas ficam na base como conferência.
A coluna "Preço mediano (R$)" da P2.9 (D58) usa a mesma medida por fornecedor; não há tabela pronta para ela, então a célula grava
os valores esperados do exemplo (dipirona 2022) e confere que, num item-ano com um só fornecedor, o preço dele é a mediana do item. Os exemplos com a
dipirona 500 mg mostram que, num período, a medida junta as compras dos anos, e não tira a média das medianas anuais.
O tamanho de uma tabela pronta por período (item × período contínuo) mostra por que a conta fica no Power BI. Grava em Auditoria/16g.

In [24]:
# 16g. P2.2 "Destaque do preço" (complemento da D46): confere a medida do Power BI contra as medianas prontas da rotina 12.
# Medida no Power BI: mediana de fato_compras[preco_real] com compra_valida = "Sim"; vazia com menos de 3 compras no filtro.
P_PAINEL = BASE / 'Base Painel'   # saída da rotina 12
MIN_MEDIANA_FILTRO = 3   # D45: mediana só com 3 preços ou mais (IN SEGES/ME 65/2021, art. 6º); na P2.2 a contagem fica no Power BI (D46)
ITEM_EX = 'BR0267203|COMPRIMIDO'   # D46: item do exemplo da página do item (dipirona 500 mg, comprimido)
EXEMPLOS_16G = [(2022, 2022, None), (2022, 2022, ['SP']), (2022, 2022, ['SP', 'PR']),
                (2018, 2020, None), (2018, 2020, ['SP']), (2009, 2023, None)]   # D46: recortes do exemplo: um ano, um e dois estados, três anos e o período inteiro
fato16 = pl.read_parquet(P_PAINEL / 'fato_compras.parquet', columns=['item', 'ano', 'uf', 'preco_real', 'mediana_item_ano', 'compra_valida'])
assert fato16['compra_valida'].is_in(['Sim', 'Não']).all(), 'Rode a rotina 12 atualizada (coluna compra_valida, D61)'
val16 = fato16.filter(pl.col('compra_valida') == 'Sim')

# (a) país, um ano: a medida = mediana pronta?
med_ia = val16.group_by(['item', 'ano']).agg(pl.col('preco_real').median().alias('medida'), pl.len().alias('compras'),
                                             pl.col('mediana_item_ano').first().alias('mediana_fato'))
disp16 = pl.read_parquet(P_PAINEL / 'ind_dispersao_item_ano.parquet', columns=['item', 'ano', 'mediana'])
c_disp = disp16.join(med_ia, on=['item', 'ano'], how='left')
# (b) estado, um ano: a medida = mediana pronta, com o mesmo número de compras? E a razão estado ÷ país (P2.5/P2.6)?
est16 = pl.read_parquet(P_PAINEL / 'ind_item_estado_ano.parquet', columns=['item', 'ano', 'uf', 'compras', 'mediana_estado', 'preco_relativo'])
med_iu = val16.group_by(['item', 'ano', 'uf']).agg(pl.col('preco_real').median().alias('medida'), pl.len().alias('compras_medida'))
c_est = (est16.join(med_iu, on=['item', 'ano', 'uf'], how='left').join(med_ia.select(['item', 'ano', 'medida']).rename({'medida': 'medida_pais'}), on=['item', 'ano'], how='left')
         .with_columns((pl.col('medida') / pl.col('medida_pais')).round(3).alias('razao_medida')))   # 3 casas, como a rotina 12
dif = lambda a, b: int(((a - b).abs() > 1e-9).sum() + a.is_null().sum() + b.is_null().sum())
conf16 = pl.DataFrame({
    'conferencia': ['país, um ano: medida × mediana_item_ano (fato_compras)', 'país, um ano: medida × ind_dispersao_item_ano',
                    'estado, um ano: medida × ind_item_estado_ano', 'estado, um ano: número de compras',
                    'estado, um ano: razão estado ÷ país × preco_relativo (P2.5/P2.6)'],
    'combinacoes': [med_ia.height, c_disp.height, c_est.height, c_est.height, c_est.height],
    'diferentes': [dif(med_ia['medida'], med_ia['mediana_fato']), dif(c_disp['medida'], c_disp['mediana']),
                   dif(c_est['medida'], c_est['mediana_estado']), int((c_est['compras'] != c_est['compras_medida'].fill_null(0)).sum()),
                   dif(c_est['razao_medida'], c_est['preco_relativo'])]})

# (c) exemplos: período junta as compras (não é a média das medianas anuais)
linhas16g = []
for ini, fim, ufs in EXEMPLOS_16G:
    g = val16.filter((pl.col('item') == ITEM_EX) & pl.col('ano').is_between(ini, fim))
    m_pais = g['preco_real'].median()   # 1,00 das barras: mediana do país no período (ignora o filtro de estado)
    if ufs: g = g.filter(pl.col('uf').is_in(ufs))
    n = g.height
    anuais = g.group_by('ano').agg(pl.col('preco_real').median())['preco_real']
    linhas16g.append({'periodo': f'{ini}' if ini == fim else f'{ini}-{fim}', 'estados': '+'.join(ufs) if ufs else 'país',
                      'compras_validas': n, 'faixa_p22_rs': round(g['preco_real'].median(), 4) if n >= MIN_MEDIANA_FILTRO else None,
                      'media_das_medianas_anuais_rs': round(anuais.mean(), 4) if ini != fim else None,
                      'barra_p26_estado_div_pais': round(g['preco_real'].median() / m_pais, 3) if ufs and len(ufs) == 1 and n >= MIN_MEDIANA_FILTRO else None})
t16g = pl.DataFrame(linhas16g)

# (d) tamanho de uma tabela pronta no Python: item × período contínuo com 3 compras válidas ou mais
cont16 = med_ia.select(['item', 'ano', 'compras'])
linhas_periodo = sum(cont16.filter(pl.col('ano').is_between(i, f)).group_by('item').agg(pl.col('compras').sum())
                     .filter(pl.col('compras') >= MIN_MEDIANA_FILTRO).height
                     for i in ANOS for f in ANOS if f >= i)

# (e) P2.9 "Quem vende", coluna "Preço mediano (R$)" (D58, item 4): a mesma medida da P2.2, por fornecedor no filtro
ANO_EX_P29 = 2022   # D58: ano do exemplo da tabela Quem vende (dipirona 500 mg, 95 fornecedores)
f16 = pl.read_parquet(P_PAINEL / 'fato_compras.parquet', columns=['item', 'ano', 'cnpj_fornecedor', 'compra_valida', 'preco_real'])
forn16 = (f16.filter(pl.col('compra_valida') == 'Sim').group_by(['item', 'ano', 'cnpj_fornecedor'])
             .agg(pl.len().alias('compras_validas'), pl.col('preco_real').median().alias('preco_mediano')))
linhas_fia = f16.select(['item', 'ano', 'cnpj_fornecedor']).unique().height
com_preco = forn16.filter(pl.col('compras_validas') >= MIN_MEDIANA_FILTRO)
# conferência interna: item-ano com um único fornecedor válido -> preço do fornecedor = mediana do item no ano
um_forn = forn16.group_by(['item', 'ano']).agg(pl.len().alias('k'), pl.col('preco_mediano').first().alias('pf')).filter(pl.col('k') == 1)
c_um = um_forn.join(med_ia.select(['item', 'ano', 'medida']), on=['item', 'ano'], how='left')
dif_um = dif(c_um['pf'], c_um['medida'])
nomes16 = pl.read_parquet(P_PAINEL / 'dim_fornecedor.parquet', columns=['cnpj_fornecedor', 'nome'])
t16g_forn = (f16.filter((pl.col('item') == ITEM_EX) & (pl.col('ano') == ANO_EX_P29)).group_by('cnpj_fornecedor').agg(pl.len().alias('compras'))
             .join(forn16.filter((pl.col('item') == ITEM_EX) & (pl.col('ano') == ANO_EX_P29)).select(['cnpj_fornecedor', 'compras_validas', 'preco_mediano']),
                   on='cnpj_fornecedor', how='left')
             .join(nomes16, on='cnpj_fornecedor', how='left')
             .with_columns(pl.col('compras_validas').fill_null(0),
                           pl.when(pl.col('compras_validas').fill_null(0) >= MIN_MEDIANA_FILTRO).then(pl.col('preco_mediano').round(4))
                             .otherwise(None).alias('p29_preco_rs'))   # vazio nunca vira zero (D45)
             .select(['nome', 'cnpj_fornecedor', 'compras', 'compras_validas', 'p29_preco_rs'])
             .sort(['compras', 'compras_validas', 'nome'], descending=[True, True, False]))   # nome desempata: mesma ordem em toda execução

print('16g. P2.2, P2.5 e P2.6 — as medidas reproduzem as medianas prontas da rotina 12?'); print(conf16)
print(f'\nExemplos ({ITEM_EX}):'); print(t16g)
print(f'\nTabela pronta por período, só país: {linhas_periodo:,} linhas (por isso a conta fica no Power BI).')
salvar(conf16, '16g_p22_conferencia')
salvar(t16g, '16g_p22_exemplos')
print(f'\nP2.9, preço por fornecedor: {com_preco.height:,} de {linhas_fia:,} linhas fornecedor-item-ano com 3 compras válidas ou mais '
      f'({com_preco["compras_validas"].sum():,} compras válidas); item-ano com um só fornecedor: {c_um.height:,}, diferentes da mediana do item: {dif_um}')
print(f'Exemplo {ITEM_EX} {ANO_EX_P29}: {t16g_forn.height} fornecedores, {t16g_forn["p29_preco_rs"].is_not_null().sum()} com preço'); print(t16g_forn.head(10))
salvar(t16g_forn, '16g_p29_preco_fornecedor_exemplo')
assert conf16['diferentes'].sum() == 0, 'a medida não reproduz as medianas prontas da rotina 12'
assert dif_um == 0, 'preço do fornecedor único difere da mediana do item no ano'
print('Conferência: com um ano, as medidas da P2.2, P2.5 e P2.6 são iguais às da rotina 12 (país e estado); a da P2.9 é coerente com a mediana do item.')

16g. P2.2, P2.5 e P2.6 — as medidas reproduzem as medianas prontas da rotina 12?
shape: (5, 3)
┌──────────────────────────────────────────────────────────────────┬─────────────┬────────────┐
│ conferencia                                                      ┆ combinacoes ┆ diferentes │
│ ---                                                              ┆ ---         ┆ ---        │
│ str                                                              ┆ i64         ┆ i64        │
╞══════════════════════════════════════════════════════════════════╪═════════════╪════════════╡
│ país, um ano: medida × mediana_item_ano (fato_compras)           ┆ 78600       ┆ 0          │
│ país, um ano: medida × ind_dispersao_item_ano                    ┆ 14188       ┆ 0          │
│ estado, um ano: medida × ind_item_estado_ano                     ┆ 65596       ┆ 0          │
│ estado, um ano: número de compras                                ┆ 65596       ┆ 0          │
│ estado, um ano: razão estado ÷ país × p

## 17. Escolha da fonte de dados e a ressalva de 2023 (D21)
**Decisão.** A base do trabalho é a **Fonte A**: as *bases anuais compiladas* do Banco de Preços em Saúde (gov.br/saude),
uma fotografia fechada de cada ano. A **Fonte B** (dados.gov.br, conjunto "Banco de Preço em Saúde – BPS", a partir de 2020)
foi avaliada e não foi usada, porque continua sendo alterada, para mais e para menos, e não repõe as compras que faltam em 2023.

**Testes (out/2026).** Arquivos guardados na pasta `Comparacao Fontes`, porque um novo download não reproduz a Fonte B:
- **17a.** Fonte A baixada em junho e de novo em outubro de 2026 (6 anos de amostra): dados idênticos. A única diferença
  é uma linha de título no topo de alguns arquivos de outubro, que em junho foi retirada à mão antes da conversão (ver README);
  em 2023, essa linha é o aviso que remete ao informe do Ministério sobre os dados de 2023 e 2024.
- **17b.** Compras por mês em 2021 e 2023 nas duas fontes: a queda de abril a dezembro de 2023 aparece também na Fonte B.
- **17c.** Ano em que cada compra entrou na Fonte B: compras de 2021 e de 2023 continuam sendo inseridas em 2026.
- **17d.** Sobreposição pela chave (comprador, código CATMAT, data): as fontes coincidem só em parte.
- **17e.** Duplicatas exatas: as que a rotina 03 retirou da Fonte A e as da Fonte B hoje (nenhuma); e a contagem de 2021
  na Fonte B registrada numa contagem anterior do trabalho (registro v5), maior que a de hoje — a Fonte B também exclui registros.

**Referências.** Ministério da Saúde, *Informe: dados de 2023 a 2024* (gov.br/saude/pt-br/acesso-a-informacao/banco-de-precos/informe-dados-de-2023-a-2024);
página das bases compiladas 2023–2024 ("Atualizado em 24/02/2025"); dados.gov.br/dados/conjuntos-dados/banco-de-preco-em-saude-bps.

A célula de código desta seção é independente: pode ser rodada sozinha, depois das células 1 e 2 (caminho dos dados e instalação), sem carregar a base analítica.

Se a pasta `Comparacao Fontes` não existir, a seção avisa e é pulada; as demais seções não dependem dela.

In [25]:
# 17. Comparação das fontes (D21). Lê os arquivos guardados na pasta 'Comparacao Fontes'.
# Os downloads não se repetem (a Fonte B muda a cada dia), por isso os arquivos ficam guardados.
# Célula independente: roda sozinha depois das células 1 e 2 (caminho e pip), sem carregar a base analítica.
from pathlib import Path
import re, time, warnings
import pandas as pd
import polars as pl

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True)

BASE = Path(PASTA_DADOS)
P_CSV = BASE / 'Base Conversao CSV'
P_FONTES = BASE / 'Comparacao Fontes'
P_A_JUN  = P_FONTES / 'Fonte A_Junho_2026'     # bases anuais compiladas (gov.br), baixadas em jun/2026
P_A_OUT  = P_FONTES / 'Fonte A_Outubro_2026'   # as mesmas, baixadas de novo em out/2026 (amostra de 6 anos)
P_B_OUT  = P_FONTES / 'Fonte B_Outubro_2026'   # dados.gov.br, baixados em 16/10/2026
ANOS_B   = [2021, 2023]                        # D21: anos baixados da Fonte B (2021 tem a contagem antiga da v5)
FONTE_B_2021_V5 = 85012                        # D21: compras de 2021 na Fonte B, registradas numa contagem anterior do trabalho (registro v5);
                                               # o arquivo daquela época não foi guardado (número só de registro)
SAIDA = BASE / 'Auditoria'

def salvar(df, nome):
    df.write_csv(SAIDA / f'{nome}.csv')
    print(f'  -> Auditoria/{nome}.csv')

faltando = [p for p in [P_A_JUN, P_A_OUT, P_B_OUT] if not p.exists()]
if faltando:
    print('SEÇÃO 17 NÃO RODADA: pasta(s) não encontrada(s):')
    for p in faltando: print('  ', p)
    print('Os resultados registrados ficam na D21. As demais seções não dependem desta.')
else:
    def ler_excel_bruto(arq):
        # a linha de cabeçalho é a primeira que contém 'Código BR'; antes dela pode haver uma linha de título
        # (presente em out/2026 e retirada à mão em jun/2026 — ver README; a de 2023 é o aviso do informe)
        # leitor 'calamine': cerca de 5 vezes mais rápido que o openpyxl, mesmo conteúdo (conferido em 16/10/2026)
        bruto = pd.read_excel(arq, header=None, dtype=object, engine='calamine')
        lin = next(i for i in range(len(bruto)) if bruto.iloc[i].astype(str).str.strip().eq('Código BR').any())
        d = bruto.iloc[lin + 1:].reset_index(drop=True)
        d.columns = [str(c) for c in bruto.iloc[lin]]
        return d.fillna('§').astype(str), lin

    # 17a. Fonte A: junho x outubro de 2026 — os dados mudaram?
    linhas = []
    for arq_j in sorted(P_A_JUN.iterdir()):
        arq_o = P_A_OUT / arq_j.name
        if not arq_o.exists():
            continue
        t0 = time.time()
        j, tj = ler_excel_bruto(arq_j)
        o, to = ler_excel_bruto(arq_o)
        print(f'  lido: {arq_j.name} (junho e outubro) em {time.time() - t0:.0f} s')
        # o arquivo usado no trabalho (Base Tratamento Minimo) também é comparado com o de
        # junho, célula a célula; o arquivo pode diferir em bytes (ex.: salvo de novo no Excel) sem diferir no conteúdo
        arq_b = BASE / 'Base Tratamento Minimo' / arq_j.name
        base_igual = (ler_excel_bruto(arq_b)[0].values.tolist() == j.values.tolist()) if arq_b.exists() else None
        linhas.append({'arquivo': arq_j.name, 'linhas_junho': len(j), 'linhas_outubro': len(o),
                       'base_do_trabalho_igual_a_junho': base_igual,
                       'base_do_trabalho_mesmo_arquivo': (arq_b.read_bytes() == arq_j.read_bytes()) if arq_b.exists() else None,
                       'linhas_titulo_junho': tj, 'linhas_titulo_outubro': to,
                       'colunas_iguais': list(j.columns) == list(o.columns),
                       'dados_identicos': j.values.tolist() == o.values.tolist()})
    t17a = pl.DataFrame(linhas)
    print('17a. Fonte A (bases anuais compiladas): download de junho x outubro de 2026'); print(t17a)
    salvar(t17a, '17a_fonte_A_junho_x_outubro')

    # Fonte A como usada no trabalho: CSV convertido (rotina 02) sem as duplicatas exatas (rotina 03, D12)
    def norm(c): return re.sub(r'\s+', ' ', str(c).replace('\ufeff', '').replace('\xa0', ' ')).strip().lower()
    def data(col):
        t = pl.col(col).str.strip_chars().str.slice(0, 10)
        return pl.coalesce(t.str.to_date('%Y-%m-%d', strict=False), t.str.to_date('%d/%m/%Y', strict=False))
    def chave(df, inst, cat, dt):
        return df.select(pl.col(inst).str.replace_all(r'\D', '').str.zfill(14).alias('inst'),
                         pl.col(cat).str.replace_all(r'\D', '').str.strip_chars_start('0').alias('cat'),
                         pl.col(dt).alias('dc'))

    t_mes, t_ins, t_sob, t_dup = [], [], [], []
    for a in ANOS_B:
        print(f'  comparando Fonte A x Fonte B: {a}')
        A = pl.read_csv(P_CSV / f'BPS_{a}.csv', infer_schema_length=0).unique(maintain_order=True)
        A = A.rename({c: norm(c) for c in A.columns})
        cA = next(c for c in A.columns if c in ('compra', 'data compra'))
        A = A.with_columns(data(cA).alias('dc')).filter(pl.col('dc').dt.year() == a)
        B = pl.read_csv(P_B_OUT / f'{a}.csv', separator=';', infer_schema_length=0)
        n_B_arquivo = B.height
        B = B.with_columns(pl.col('dt_compra').str.to_date('%d/%m/%Y', strict=False).alias('dc'),
                           pl.col('dt_insercao').str.to_date('%d/%m/%Y', strict=False).alias('di')
                           ).filter(pl.col('dc').dt.year() == a)
        # 17b. compras por mês
        mA = A.group_by(pl.col('dc').dt.month().cast(pl.Int32).alias('mes')).agg(pl.len().alias('fonte_A'))
        mB = B.group_by(pl.col('dc').dt.month().cast(pl.Int32).alias('mes')).agg(pl.len().alias('fonte_B'))
        t_mes.append(mA.join(mB, on='mes', how='full', coalesce=True).with_columns(pl.lit(a).alias('ano')))
        # 17c. ano em que a compra foi inserida na Fonte B
        t_ins.append(B.group_by(pl.col('di').dt.year().alias('ano_insercao')).agg(pl.len().alias('compras'))
                      .with_columns(pl.lit(a).alias('ano_compra'), pl.lit(str(B['di'].max())).alias('ultima_insercao')))
        # 17d. sobreposição pela chave (comprador, código CATMAT, data da compra)
        kA = chave(A, next(c for c in A.columns if 'cnpj' in c and 'institui' in c),
                   next(c for c in A.columns if c.startswith('código br')), 'dc').unique()
        kB = chave(B, 'cnpj_instituicao', 'co_catmat', 'dc').unique()
        comum = kA.join(kB, on=['inst', 'cat', 'dc'], how='inner').height
        t_sob.append({'ano': a, 'compras_A': A.height, 'compras_B': B.height, 'chaves_A': kA.height,
                      'chaves_B': kB.height, 'chaves_em_comum': comum,
                      'pct_de_A_presente_em_B': round(100 * comum / kA.height, 1),
                      'pct_de_B_presente_em_A': round(100 * comum / kB.height, 1)})
        # 17e. duplicatas: na Fonte B hoje x as retiradas da Fonte A pela rotina 03
        Bb = pl.read_csv(P_B_OUT / f'{a}.csv', separator=';', infer_schema_length=0)
        sem_id = [c for c in Bb.columns if c not in ('co_seq_bps', 'dt_insercao')]
        A_bruto = pl.read_csv(P_CSV / f'BPS_{a}.csv', infer_schema_length=0)
        t_dup.append({'ano': a, 'linhas_A_bruto': A_bruto.height,
                      'duplicatas_exatas_A': A_bruto.height - A_bruto.unique().height,
                      'linhas_B_out2026': n_B_arquivo,
                      'duplicatas_exatas_B': Bb.height - Bb.unique().height,
                      'duplicatas_B_exceto_id_e_insercao': Bb.height - Bb.unique(subset=sem_id).height,
                      'compras_B_registro_v5': FONTE_B_2021_V5 if a == 2021 else None})

    t17b = pl.concat(t_mes).select(['ano', 'mes', 'fonte_A', 'fonte_B']).sort(['ano', 'mes'])
    t17b = t17b.with_columns(pl.when(pl.col('mes') <= 3).then(pl.lit('jan-mar')).otherwise(pl.lit('abr-dez')).alias('periodo'))
    print('\n17b. Compras por mês: Fonte A (base do trabalho) x Fonte B (dados.gov.br, 16/10/2026)'); print(t17b)
    print(t17b.group_by(['ano', 'periodo']).agg(pl.col('fonte_A').sum(), pl.col('fonte_B').sum()).sort(['ano', 'periodo']))
    salvar(t17b, '17b_fonte_A_x_B_por_mes')

    t17c = pl.concat(t_ins).select(['ano_compra', 'ano_insercao', 'compras', 'ultima_insercao']).sort(['ano_compra', 'ano_insercao'])
    print('\n17c. Fonte B: ano em que cada compra entrou no sistema'); print(t17c)
    salvar(t17c, '17c_fonte_B_ano_insercao')

    t17d = pl.DataFrame(t_sob)
    print('\n17d. Sobreposição entre as fontes (comprador, código CATMAT, data)'); print(t17d)
    salvar(t17d, '17d_fonte_A_x_B_sobreposicao')

    t17e = pl.DataFrame(t_dup)
    print('\n17e. Duplicatas exatas: Fonte A (retiradas pela rotina 03) x Fonte B hoje'); print(t17e)
    salvar(t17e, '17e_fonte_A_x_B_duplicatas')

    # conferências: a Fonte A não mudou; a Fonte B segue sem o 2º–4º trimestre de 2023
    assert t17a['dados_identicos'].all(), 'A Fonte A mudou entre junho e outubro: revisar a D21'
    assert t17a['base_do_trabalho_igual_a_junho'].all(), 'A base do trabalho difere do download de junho: revisar a D21'
    print('\nConferência: Fonte A idêntica nos', t17a.height, 'arquivos comparados.')

  lido: BPS_2009.xls (junho e outubro) em 2 s
  lido: BPS_2012.xls (junho e outubro) em 1 s
  lido: BPS_2015.xlsx (junho e outubro) em 14 s
  lido: BPS_2018.xlsx (junho e outubro) em 19 s
  lido: BPS_2021.xlsx (junho e outubro) em 10 s
  lido: BPS_2023.xlsx (junho e outubro) em 5 s
17a. Fonte A (bases anuais compiladas): download de junho x outubro de 2026
shape: (6, 9)
┌───────────────┬──────────────┬────────────────┬────────────────────────────────┬────────────────────────────────┬─────────────────────┬───────────────────────┬────────────────┬─────────────────┐
│ arquivo       ┆ linhas_junho ┆ linhas_outubro ┆ base_do_trabalho_igual_a_junho ┆ base_do_trabalho_mesmo_arquivo ┆ linhas_titulo_junho ┆ linhas_titulo_outubro ┆ colunas_iguais ┆ dados_identicos │
│ ---           ┆ ---          ┆ ---            ┆ ---                            ┆ ---                            ┆ ---                 ┆ ---                   ┆ ---            ┆ ---             │
│ str           ┆ i64          ┆ i64

## Arquivos gerados

In [26]:
for f in sorted(SAIDA.iterdir()):
    print(' ', f.name)

  00a_datas_da_compra.csv
  00b_separador_decimal.csv
  00c_valor_total.csv
  00d_colunas_cnpj.csv
  00e_codigo_catmat.csv
  00f_marca_generico_na_unidade.csv
  00g_linhas_sem_preco_e_descricao.csv
  00h_catmat_6_digitos_reconexao.csv
  01_esfera_bps_x_natureza.csv
  02_sem_cnpj_por_ano.csv
  03_cobertura_uf_municipio.csv
  03a_validacao_uf_cnpj.csv
  03b_validacao_uf_receita.csv
  03c_preenchidos_receita.csv
  03d_classe_bps_x_catalogo.csv
  03e_divergencias_classe.csv
  03f_cobertura_classe_por_ano.csv
  03g_codigos_fora_classes_saude.csv
  04a_20_maiores_registros.csv
  04b_sensibilidade_regua.csv
  04c_alcance_da_regua.csv
  05_sensibilidade_regua_hhi_valor.csv
  06a_prova_minimo_7.csv
  06b_sensibilidade_minimo.csv
  06c_faixas_fornecedores_d33.csv
  06d_robustez_item_catmat_d17.csv
  07a_sensibilidade_indice.csv
  07b_formato_unidade_por_ano.csv
  07c2_decomposicao_elos_min3.csv
  07c_decomposicao_elos.csv
  08_escala_chave_item.csv
  09_territorial_estabilidade.csv
  09b_sensibi